# Boundary-aware probabilistic cell tracking with global-relative motion

This notebook tracks cells in physical `(Z, Y, X)` space while separating
population-wide motion from cell-specific motion and treating association as a
competition among three explicit outcomes:

1. continue an existing track with a current detection;
2. leave the track temporarily unmatched;
3. create a new track from an unmatched detection.

The motion model remains

\[
\text{cell displacement}
=
\text{global frame displacement}
+
\text{cell-relative residual displacement}.
\]

The association model replaces the previous narrow distance and volume hard
gates with smooth, heavy-tailed likelihood costs. Only broad physical safety
gates remain to remove impossible candidates. Hungarian assignment is applied
to an augmented matrix containing pair, miss, and birth alternatives, so the
optimizer is never forced either to match an implausible pair or to break a
plausible pair solely because it lies slightly beyond an arbitrary threshold.

The resulting association probabilities are model-relative confidence values,
not yet empirically calibrated probabilities. Candidate-level diagnostics are
saved so that their scales and priors can later be calibrated from curated
tracking examples.


## Probabilistic-association implementation plan

### 1. Preserve the global-relative motion predictor

For track \(i\) at frame \(t\),

\[
\Delta \mathbf{p}_{i,t}
=
\mathbf{g}_t + \mathbf{r}_{i,t},
\]

where \(\mathbf{g}_t\) is the robust frame-wide shift and
\(\mathbf{r}_{i,t}\) is the confidence-weighted cell-relative residual
velocity. Cumulative global shifts remain the prediction anchor across missing
frames.

### 2. Replace narrow hard gates with smooth likelihood costs

For every track-detection pair, compute independent evidence terms:

- a 3-D Student-t position cost based on an adaptive Mahalanobis residual;
- a heavy-tailed log-volume-ratio cost;
- robust size, shape, intensity, and bounding-box costs;
- a relative-motion consistency cost;
- a boundary-face compatibility cost.

The position uncertainty grows with missing-frame gaps, low global-shift
confidence, boundary truncation, and the track's learned prediction residual.
Volume and appearance changes are penalized smoothly rather than becoming
impossible immediately after a threshold.

### 3. Retain only broad physical safety gates

Extremely distant pairs and physically absurd volume changes are still removed.
These limits are deliberately much wider than the previous operational gates;
they exist only to prevent unrelated detections from competing and to keep the
assignment numerically stable.

### 4. Add explicit miss and birth alternatives

The assignment matrix is augmented with:

- one private miss column for every track;
- one private birth row for every detection;
- zero-cost dummy-to-dummy assignments.

Miss and birth costs are derived from configurable context-dependent priors.
Boundary tracks may disappear more easily, and boundary detections may enter as
new tracks more easily, than interior observations.

### 5. Convert costs to auditable model probabilities

For each pair, calculate a track-normalized continuation probability and a
detection-normalized ownership probability relative to the corresponding miss
or birth option. Their geometric mean is stored as the association confidence.
The best alternative and probability margin are also recorded.

### 6. Allow conservative interior recovery

A selected miss no longer destroys an interior track immediately. Interior
tracks receive one configurable recovery frame; boundary tracks retain their
longer boundary-specific memory. This lets a high-confidence later observation
repair one temporary segmentation failure.

### 7. Protect learned motion and uncertainty

Relative velocity and per-track prediction uncertainty are updated only from
selected, immediate-frame, sufficiently confident matches with an adequate
probability margin. Ambiguous matches cannot easily corrupt future prediction.

### 8. Preserve detailed diagnostics

The notebook saves transition summaries, match/miss/birth decisions, and the
top candidate pairs for every track. These diagnostics expose whether a break
was caused by position evidence, volume evidence, appearance evidence,
competition, or the explicit no-match decision.


In [ ]:
from pathlib import Path
import json

import numpy as np
import pandas as pd

from scipy.optimize import linear_sum_assignment
from scipy.spatial.distance import cdist


In [ ]:
# ------------------------------------------------------------
# Paths and sample configuration
# ------------------------------------------------------------

PROJECT_ROOT = Path.cwd().parent

DATA_ROOT = PROJECT_ROOT / "data/sample"

SAMPLE_ID = "44b6_0113de3b"

PROCESSED_DIR = (
    DATA_ROOT
    / "processed"
    / "stage_6_processed_dataset"
    / SAMPLE_ID
)

FEATURE_DIR = PROCESSED_DIR / "cells"

OUTPUT_DIR = (
    DATA_ROOT
    / "processed"
    / "stage_7_cell_tracking"
)

# Volume geometry for this dataset, in (Z, Y, X).
VOLUME_SHAPE_ZYX = np.asarray(
    [64, 256, 256],
    dtype=int,
)

VOXEL_SIZE_ZYX = np.asarray(
    [1.625, 0.40625, 0.40625],
    dtype=float,
)


In [ ]:
# ------------------------------------------------------------
# Load per-frame cell detections
# ------------------------------------------------------------

cell_files = sorted(FEATURE_DIR.glob("t*.csv"))

if not cell_files:
    raise FileNotFoundError(
        f"No per-frame cell CSV files were found in:\n{FEATURE_DIR}"
    )

time_frames = [
    pd.read_csv(file)
    for file in cell_files
]

print(f"Loaded {len(time_frames)} timepoints.")


In [ ]:
time_frames[0].head()

In [ ]:
for t, df in enumerate(time_frames):
    print(f"t={t:03d}: {len(df)} cells")


In [ ]:
# ============================================================
# Probabilistic global-relative tracking configuration
# ============================================================

# ------------------------------------------------------------
# Robust global-shift estimation
# ------------------------------------------------------------

GLOBAL_SHIFT_MAX_PAIR_DISTANCE_UM = 12.0
GLOBAL_SHIFT_MAD_SCALE = 3.5
GLOBAL_SHIFT_MIN_INLIER_RADIUS_UM = 1.0
GLOBAL_SHIFT_CONFIDENCE_PAIR_COUNT = 40
GLOBAL_SHIFT_CONFIDENCE_DISPERSION_UM = 2.0

# Second-pass global-shift refinement now uses association
# probability rather than the previous normalized-cost threshold.
GLOBAL_REFINEMENT_MIN_MATCHES = 20
GLOBAL_REFINEMENT_MIN_ASSOCIATION_PROBABILITY = 0.20
GLOBAL_REFINEMENT_MIN_PROBABILITY_MARGIN = 0.02
GLOBAL_REFINEMENT_MAX_PREDICTION_ERROR_UM = 3.5
GLOBAL_REFINEMENT_OBJECTIVE_TOLERANCE = 0.02
GLOBAL_REFINEMENT_MAX_MATCH_LOSS = 2

# ------------------------------------------------------------
# Global-relative motion state
# ------------------------------------------------------------

RELATIVE_VELOCITY_EMA_ALPHA = 0.50
RELATIVE_ERROR_EMA_ALPHA = 0.25
RELATIVE_FULL_CONFIDENCE_SAMPLES = 4
RELATIVE_ERROR_CONFIDENCE_SCALE_UM = 2.0
RELATIVE_MOTION_GAP_DECAY = 0.65
BOUNDARY_RELATIVE_MOTION_CONFIDENCE_SCALE = 0.35
RELATIVE_MOTION_COST_SCALE_UM = 3.0

# Only confident immediate-frame matches may update relative
# velocity and per-track prediction uncertainty.
RELATIVE_UPDATE_MIN_GLOBAL_CONFIDENCE = 0.20
RELATIVE_UPDATE_MIN_ASSOCIATION_PROBABILITY = 0.20
RELATIVE_UPDATE_MIN_PROBABILITY_MARGIN = 0.02
RELATIVE_UPDATE_MAX_DISTANCE_UM = 5.5
RELATIVE_UPDATE_MAX_PAIR_COST = 6.0
MAX_RELATIVE_VELOCITY_UM_PER_FRAME = 5.0

POSITION_RESIDUAL_EMA_ALPHA = 0.20
POSITION_RESIDUAL_UPDATE_MIN_ASSOCIATION_PROBABILITY = 0.20
POSITION_RESIDUAL_UPDATE_MIN_MARGIN = 0.02

# ------------------------------------------------------------
# Adaptive position likelihood
# ------------------------------------------------------------

# Physical uncertainty is axis-specific because centroid estimates
# are less precise along the coarser Z direction.
BASE_POSITION_SIGMA_ZYX_UM = np.asarray(
    [2.40, 1.60, 1.60],
    dtype=float,
)

POSITION_SIGMA_GLOBAL_UNCERTAINTY_UM = 1.50
POSITION_SIGMA_PER_MISSING_FRAME_UM = 1.25
POSITION_SIGMA_BOUNDARY_SCALE = 1.50
POSITION_SIGMA_TRACK_RESIDUAL_WEIGHT = 0.75
POSITION_STUDENT_T_DOF = 4.0

# ------------------------------------------------------------
# Smooth volume and appearance likelihoods
# ------------------------------------------------------------

# Log-ratio scales. A scale of log(1.35), for example, treats an
# approximately 35% change as meaningful but not impossible.
VOLUME_LOG_SCALE_INTERIOR = float(np.log(1.25))
VOLUME_LOG_SCALE_BOUNDARY = float(np.log(2.50))
VOLUME_STUDENT_T_DOF = 8.0

SIZE_RELATIVE_SCALE = 0.30
SHAPE_RELATIVE_SCALE = 0.25
INTENSITY_RELATIVE_SCALE = 0.30
BBOX_RELATIVE_SCALE = 0.35
FEATURE_STUDENT_T_DOF = 4.0

# The scales above control tolerance; these weights temper the
# relative influence of evidence groups in the pair negative-log score.
INTERIOR_W_POSITION = 1.00
INTERIOR_W_VOLUME = 1.25
INTERIOR_W_SIZE = 0.35
INTERIOR_W_SHAPE = 0.55
INTERIOR_W_INTENSITY = 0.30
INTERIOR_W_BBOX = 0.20
INTERIOR_W_MOTION = 0.15

BOUNDARY_W_POSITION = 1.00
BOUNDARY_W_VOLUME = 0.10
BOUNDARY_W_MOTION = 0.25
BOUNDARY_W_INTENSITY = 0.25
BOUNDARY_W_FACE = 0.25

# ------------------------------------------------------------
# Explicit miss and birth priors
# ------------------------------------------------------------

# These are context priors used to construct -log(probability)
# alternatives in the augmented assignment matrix.
MISS_PROBABILITY_INTERIOR = 0.08
MISS_PROBABILITY_BOUNDARY = 0.30
MISS_PROBABILITY_PENDING_INTERIOR = 0.35
MISS_PROBABILITY_PENDING_BOUNDARY = 0.55
MISS_GLOBAL_UNCERTAINTY_BONUS = 0.15

BIRTH_PROBABILITY_INTERIOR = 0.06
BIRTH_PROBABILITY_BOUNDARY = 0.30

# Interior tracks receive one recovery frame after a selected miss.
INTERIOR_MAX_MISSING_FRAMES = 1
BOUNDARY_MAX_MISSING_FRAMES = 2

# ------------------------------------------------------------
# Broad physical safety gates
# ------------------------------------------------------------

# These are not operational association thresholds. They only remove
# physically absurd pairs after soft likelihoods have been computed.
ABSOLUTE_MAX_DISTANCE_INTERIOR_UM = 18.0
ABSOLUTE_MAX_DISTANCE_BOUNDARY_UM = 28.0
ABSOLUTE_DISTANCE_PER_MISSING_FRAME_UM = 5.0

ABSOLUTE_MAX_VOLUME_RATIO_INTERIOR = 8.0
ABSOLUTE_MAX_VOLUME_RATIO_BOUNDARY = 20.0

# ------------------------------------------------------------
# Boundary metadata and diagnostics
# ------------------------------------------------------------

BOUNDARY_MARGIN_UM = 2.0
TEMPLATE_EMA_ALPHA = 0.20

SAVE_TOP_ASSOCIATION_CANDIDATES = True
TOP_ASSOCIATION_CANDIDATES_PER_TRACK = 5

PROBABILITY_FLOOR = 1e-9
INVALID_COST = 1e6
EPS = 1e-8

print(
    "Boundary margin in voxels:",
    np.ceil(
        BOUNDARY_MARGIN_UM / VOXEL_SIZE_ZYX
    ).astype(int),
)


In [ ]:
# ============================================================
# Boundary metadata, motion modelling, and assignment helpers
# ============================================================

SIZE_FEATURES = [
    "extent",
    "equivalent_radius",
]

SHAPE_FEATURES = [
    "elongation",
    "flatness",
    "anisotropy",
    "solidity",
    "compactness",
]

INTENSITY_FEATURES = [
    "intensity_mean",
    "intensity_std",
    "intensity_cv",
]

BBOX_FEATURES = [
    "bbox_depth",
    "bbox_height",
    "bbox_width",
]

TEMPLATE_FEATURES = sorted(
    set(
        SIZE_FEATURES
        + SHAPE_FEATURES
        + INTENSITY_FEATURES
        + BBOX_FEATURES
    )
)


def annotate_boundary_metadata(
    detections: pd.DataFrame,
) -> pd.DataFrame:
    """Add boundary-contact information to one frame's detections."""

    required_bbox_columns = {
        "z_min", "z_max",
        "y_min", "y_max",
        "x_min", "x_max",
    }

    missing = required_bbox_columns.difference(
        detections.columns
    )

    if missing:
        raise KeyError(
            "Boundary-aware tracking requires bounding-box columns. "
            f"Missing: {sorted(missing)}"
        )

    result = detections.copy()

    margin_zyx = np.ceil(
        BOUNDARY_MARGIN_UM / VOXEL_SIZE_ZYX
    ).astype(int)

    z_size, y_size, x_size = VOLUME_SHAPE_ZYX
    z_margin, y_margin, x_margin = margin_zyx

    # The stored bbox maxima are treated as upper/exclusive bounds.
    result["touches_z_min"] = result["z_min"] <= z_margin
    result["touches_z_max"] = result["z_max"] >= (
        z_size - z_margin
    )

    result["touches_y_min"] = result["y_min"] <= y_margin
    result["touches_y_max"] = result["y_max"] >= (
        y_size - y_margin
    )

    result["touches_x_min"] = result["x_min"] <= x_margin
    result["touches_x_max"] = result["x_max"] >= (
        x_size - x_margin
    )

    face_columns = [
        "touches_z_min",
        "touches_z_max",
        "touches_y_min",
        "touches_y_max",
        "touches_x_min",
        "touches_x_max",
    ]

    result["touches_boundary"] = (
        result[face_columns].any(axis=1)
    )

    result["boundary_face_count"] = (
        result[face_columns].sum(axis=1).astype(int)
    )

    face_names = [
        "z_min", "z_max",
        "y_min", "y_max",
        "x_min", "x_max",
    ]

    result["boundary_faces"] = [
        "|".join(
            face_name
            for face_name, flag in zip(
                face_names,
                flags,
            )
            if bool(flag)
        )
        for flags in result[face_columns].to_numpy()
    ]

    centroids = result[
        ["centroid_z", "centroid_y", "centroid_x"]
    ].to_numpy(dtype=float)

    lower_distance_um = (
        centroids * VOXEL_SIZE_ZYX
    )

    upper_distance_um = (
        (
            VOLUME_SHAPE_ZYX
            - 1
            - centroids
        )
        * VOXEL_SIZE_ZYX
    )

    result["distance_to_boundary_um"] = np.min(
        np.concatenate(
            [lower_distance_um, upper_distance_um],
            axis=1,
        ),
        axis=1,
    )

    result["is_boundary_partial"] = (
        result["touches_boundary"]
    )

    return result


def parse_boundary_faces(value) -> set[str]:
    """Convert a pipe-separated boundary-face string into a set."""

    if value is None or pd.isna(value):
        return set()

    text = str(value).strip()

    if not text:
        return set()

    return {
        face
        for face in text.split("|")
        if face
    }


def physical_coordinates(
    detections: pd.DataFrame,
) -> np.ndarray:
    """Return centroid coordinates in physical (Z, Y, X) units."""

    coords_voxel = detections[
        ["centroid_z", "centroid_y", "centroid_x"]
    ].to_numpy(dtype=float)

    return coords_voxel * VOXEL_SIZE_ZYX


def vector_angle_degrees(
    first: np.ndarray,
    second: np.ndarray,
) -> float:
    """Return the angle between two vectors in degrees."""

    first = np.asarray(first, dtype=float)
    second = np.asarray(second, dtype=float)

    denominator = (
        np.linalg.norm(first)
        * np.linalg.norm(second)
    )

    if denominator <= EPS:
        return np.nan

    cosine = float(
        np.dot(first, second) / denominator
    )

    return float(
        np.degrees(
            np.arccos(
                np.clip(cosine, -1.0, 1.0)
            )
        )
    )


def robust_displacement_summary(
    displacements: np.ndarray,
) -> dict:
    """Robustly summarize a collection of 3-D displacement vectors."""

    displacements = np.asarray(
        displacements,
        dtype=float,
    ).reshape(-1, 3)

    if len(displacements) == 0:
        return {
            "shift": np.zeros(3, dtype=float),
            "inlier_mask": np.zeros(0, dtype=bool),
            "inlier_count": 0,
            "dispersion_um": np.nan,
            "inlier_radius_um": np.nan,
            "confidence": 0.0,
        }

    initial_center = np.median(
        displacements,
        axis=0,
    )

    residual_norms = np.linalg.norm(
        displacements - initial_center[None, :],
        axis=1,
    )

    residual_median = float(
        np.median(residual_norms)
    )

    residual_mad = float(
        np.median(
            np.abs(
                residual_norms - residual_median
            )
        )
    )

    robust_sigma = 1.4826 * residual_mad

    inlier_radius_um = max(
        GLOBAL_SHIFT_MIN_INLIER_RADIUS_UM,
        (
            residual_median
            + GLOBAL_SHIFT_MAD_SCALE
            * robust_sigma
        ),
    )

    inlier_mask = (
        residual_norms <= inlier_radius_um
    )

    if not np.any(inlier_mask):
        inlier_mask = np.ones(
            len(displacements),
            dtype=bool,
        )

    inlier_displacements = displacements[
        inlier_mask
    ]

    shift = np.median(
        inlier_displacements,
        axis=0,
    )

    dispersion_um = float(
        np.median(
            np.linalg.norm(
                inlier_displacements
                - shift[None, :],
                axis=1,
            )
        )
    )

    pair_confidence = min(
        len(inlier_displacements)
        / max(
            GLOBAL_SHIFT_CONFIDENCE_PAIR_COUNT,
            1,
        ),
        1.0,
    )

    dispersion_confidence = float(
        np.exp(
            -dispersion_um
            / max(
                GLOBAL_SHIFT_CONFIDENCE_DISPERSION_UM,
                EPS,
            )
        )
    )

    confidence = float(
        np.clip(
            pair_confidence
            * dispersion_confidence,
            0.0,
            1.0,
        )
    )

    return {
        "shift": shift.astype(float),
        "inlier_mask": inlier_mask,
        "inlier_count": int(
            inlier_mask.sum()
        ),
        "dispersion_um": dispersion_um,
        "inlier_radius_um": float(
            inlier_radius_um
        ),
        "confidence": confidence,
    }


def estimate_global_shift_physical(
    previous_positions: np.ndarray,
    current_positions: np.ndarray,
) -> dict:
    """Estimate current global displacement using mutual nearest neighbours."""

    previous_positions = np.asarray(
        previous_positions,
        dtype=float,
    ).reshape(-1, 3)

    current_positions = np.asarray(
        current_positions,
        dtype=float,
    ).reshape(-1, 3)

    if (
        len(previous_positions) == 0
        or len(current_positions) == 0
    ):
        return {
            "shift": np.zeros(3, dtype=float),
            "method": "no_data",
            "pair_count": 0,
            "inlier_count": 0,
            "dispersion_um": np.nan,
            "inlier_radius_um": np.nan,
            "confidence": 0.0,
        }

    distances = cdist(
        previous_positions,
        current_positions,
    )

    nearest_current = np.argmin(
        distances,
        axis=1,
    )

    nearest_previous = np.argmin(
        distances,
        axis=0,
    )

    matched_displacements = []

    for previous_index, current_index in enumerate(
        nearest_current
    ):
        if (
            nearest_previous[current_index]
            != previous_index
        ):
            continue

        if (
            distances[
                previous_index,
                current_index,
            ]
            > GLOBAL_SHIFT_MAX_PAIR_DISTANCE_UM
        ):
            continue

        matched_displacements.append(
            current_positions[current_index]
            - previous_positions[previous_index]
        )

    if matched_displacements:
        matched_displacements = np.asarray(
            matched_displacements,
            dtype=float,
        )

        summary = robust_displacement_summary(
            matched_displacements
        )

        return {
            "shift": summary["shift"],
            "method": "mutual_nearest_neighbour",
            "pair_count": int(
                len(matched_displacements)
            ),
            "inlier_count": summary[
                "inlier_count"
            ],
            "dispersion_um": summary[
                "dispersion_um"
            ],
            "inlier_radius_um": summary[
                "inlier_radius_um"
            ],
            "confidence": summary[
                "confidence"
            ],
        }

    # Low-confidence fallback that still responds to a sudden direction
    # change instead of blindly repeating the previous frame's shift.
    centroid_shift = (
        np.median(current_positions, axis=0)
        - np.median(previous_positions, axis=0)
    )

    return {
        "shift": centroid_shift.astype(float),
        "method": "median_centroid_fallback",
        "pair_count": 0,
        "inlier_count": 0,
        "dispersion_um": np.nan,
        "inlier_radius_um": np.nan,
        "confidence": 0.05,
    }


def make_feature_template(
    detection: pd.Series,
) -> dict[str, float]:
    """Create a numeric feature template from one detection."""

    template = {}

    for feature in TEMPLATE_FEATURES:
        if feature not in detection.index:
            continue

        value = detection[feature]

        if pd.notna(value):
            template[feature] = float(value)

    return template


def update_feature_template(
    state: dict,
    detection: pd.Series,
) -> None:
    """Update a reliable template using only fully visible detections."""

    if bool(detection["touches_boundary"]):
        return

    values = make_feature_template(detection)

    if not state["template_reliable"]:
        state["template"] = values
        state["template_reliable"] = True
        state["template_count"] = 1
        return

    for feature, value in values.items():
        old_value = state["template"].get(
            feature,
            value,
        )

        state["template"][feature] = (
            (1.0 - TEMPLATE_EMA_ALPHA)
            * old_value
            + TEMPLATE_EMA_ALPHA
            * value
        )

    state["template_count"] += 1


def make_track_state(
    *,
    track_id: int,
    frame: int,
    detection: pd.Series,
) -> dict:
    """Create the persistent state for one new track."""

    position_voxel = detection[
        ["centroid_z", "centroid_y", "centroid_x"]
    ].to_numpy(dtype=float)

    position_physical = (
        position_voxel * VOXEL_SIZE_ZYX
    )

    is_boundary = bool(
        detection["touches_boundary"]
    )

    return {
        "track_id": int(track_id),
        "active": True,
        "last_frame": int(frame),
        "last_position_voxel": position_voxel,
        "last_position_physical": position_physical,
        "previous_position_physical": None,
        "relative_velocity_physical": np.zeros(
            3,
            dtype=float,
        ),
        "relative_velocity_valid": False,
        "relative_velocity_samples": 0,
        "relative_velocity_error_ema": 0.0,
        "relative_velocity_updates_rejected": 0,
        "last_relative_update_frame": None,
        "position_residual_ema_zyx": np.zeros(
            3,
            dtype=float,
        ),
        "position_residual_samples": 0,
        "last_detection": detection.to_dict(),
        "missed_frames": 0,
        "boundary_pending": is_boundary,
        "interior_pending": False,
        "last_boundary_faces": parse_boundary_faces(
            detection["boundary_faces"]
        ),
        "template": make_feature_template(detection),
        "template_reliable": not is_boundary,
        "template_count": 1 if not is_boundary else 0,
    }

def state_reference_value(
    state: dict,
    feature: str,
) -> float:
    """Read a feature from the reliable template or last observation."""

    if (
        state["template_reliable"]
        and feature in state["template"]
    ):
        return float(state["template"][feature])

    value = state["last_detection"].get(
        feature,
        np.nan,
    )

    return float(value) if pd.notna(value) else np.nan


def cumulative_global_displacement(
    *,
    start_frame: int,
    target_frame: int,
    global_shift_history: dict[int, np.ndarray],
) -> np.ndarray:
    """Sum frame-to-frame global shifts from start_frame to target_frame."""

    if target_frame <= start_frame:
        return np.zeros(3, dtype=float)

    missing_frames = [
        frame
        for frame in range(
            int(start_frame) + 1,
            int(target_frame) + 1,
        )
        if frame not in global_shift_history
    ]

    if missing_frames:
        raise KeyError(
            "Global-shift history is incomplete for frames "
            f"{missing_frames}."
        )

    return np.sum(
        np.vstack(
            [
                global_shift_history[frame]
                for frame in range(
                    int(start_frame) + 1,
                    int(target_frame) + 1,
                )
            ]
        ),
        axis=0,
    )


def relative_motion_confidence(
    state: dict,
    frame_gap: int,
) -> float:
    """Return confidence in a track's relative-motion correction."""

    if not state["relative_velocity_valid"]:
        return 0.0

    sample_confidence = min(
        state["relative_velocity_samples"]
        / max(
            RELATIVE_FULL_CONFIDENCE_SAMPLES,
            1,
        ),
        1.0,
    )

    error_confidence = float(
        np.exp(
            -state["relative_velocity_error_ema"]
            / max(
                RELATIVE_ERROR_CONFIDENCE_SCALE_UM,
                EPS,
            )
        )
    )

    gap_confidence = (
        RELATIVE_MOTION_GAP_DECAY
        ** max(int(frame_gap) - 1, 0)
    )

    boundary_confidence = 1.0

    if (
        state["boundary_pending"]
        or bool(
            state["last_detection"].get(
                "touches_boundary",
                False,
            )
        )
    ):
        boundary_confidence = (
            BOUNDARY_RELATIVE_MOTION_CONFIDENCE_SCALE
        )

    return float(
        np.clip(
            sample_confidence
            * error_confidence
            * gap_confidence
            * boundary_confidence,
            0.0,
            1.0,
        )
    )


def predict_state_components(
    *,
    state: dict,
    target_frame: int,
    global_shift_history: dict[int, np.ndarray],
) -> dict:
    """Return global, relative, and final prediction components."""

    frame_gap = (
        int(target_frame)
        - int(state["last_frame"])
    )

    if frame_gap <= 0:
        return {
            "predicted_position": state[
                "last_position_physical"
            ].copy(),
            "global_displacement": np.zeros(
                3,
                dtype=float,
            ),
            "relative_displacement": np.zeros(
                3,
                dtype=float,
            ),
            "relative_weight": 0.0,
            "frame_gap": frame_gap,
        }

    global_displacement = (
        cumulative_global_displacement(
            start_frame=state["last_frame"],
            target_frame=target_frame,
            global_shift_history=global_shift_history,
        )
    )

    relative_weight = (
        relative_motion_confidence(
            state,
            frame_gap,
        )
    )

    relative_displacement = (
        relative_weight
        * state["relative_velocity_physical"]
        * frame_gap
    )

    predicted_position = (
        state["last_position_physical"]
        + global_displacement
        + relative_displacement
    )

    return {
        "predicted_position": predicted_position,
        "global_displacement": global_displacement,
        "relative_displacement": relative_displacement,
        "relative_weight": relative_weight,
        "frame_gap": frame_gap,
    }


def predict_state_position(
    state: dict,
    target_frame: int,
    global_shift_history: dict[int, np.ndarray],
) -> np.ndarray:
    """Predict a track position using global plus relative motion."""

    return predict_state_components(
        state=state,
        target_frame=target_frame,
        global_shift_history=global_shift_history,
    )["predicted_position"]


def normalized_pairwise_cost(
    previous_values: np.ndarray,
    current_values: np.ndarray,
) -> np.ndarray:
    """Relative absolute difference with robust NaN handling."""

    previous_values = np.asarray(
        previous_values,
        dtype=float,
    )

    current_values = np.asarray(
        current_values,
        dtype=float,
    )

    cost = (
        np.abs(
            previous_values[:, None]
            - current_values[None, :]
        )
        / (
            np.maximum(
                np.abs(previous_values[:, None]),
                np.abs(current_values[None, :]),
            )
            + EPS
        )
    )

    return np.nan_to_num(
        cost,
        nan=1.0,
        posinf=1.0,
        neginf=1.0,
    )


def feature_group_cost(
    states: list[dict],
    detections: pd.DataFrame,
    feature_names: list[str],
) -> np.ndarray:
    """Average relative cost over available features in one group."""

    costs = []

    for feature in feature_names:
        if feature not in detections.columns:
            continue

        previous_values = np.asarray(
            [
                state_reference_value(
                    state,
                    feature,
                )
                for state in states
            ],
            dtype=float,
        )

        current_values = detections[
            feature
        ].to_numpy(dtype=float)

        costs.append(
            normalized_pairwise_cost(
                previous_values,
                current_values,
            )
        )

    if not costs:
        return np.zeros(
            (len(states), len(detections)),
            dtype=float,
        )

    return np.mean(
        np.stack(costs, axis=0),
        axis=0,
    )


def relative_motion_consistency_cost(
    *,
    states: list[dict],
    current_positions: np.ndarray,
    current_frame: int,
    global_shift_history: dict[int, np.ndarray],
    prediction_components: list[dict],
) -> np.ndarray:
    """Compare candidate residual motion after removing global motion."""

    result = np.zeros(
        (len(states), len(current_positions)),
        dtype=float,
    )

    for state_index, state in enumerate(states):
        confidence = prediction_components[
            state_index
        ]["relative_weight"]

        if confidence <= EPS:
            continue

        frame_gap = (
            current_frame
            - state["last_frame"]
        )

        global_displacement = (
            prediction_components[
                state_index
            ]["global_displacement"]
        )

        expected_relative_displacement = (
            prediction_components[
                state_index
            ]["relative_displacement"]
        )

        candidate_relative_displacements = (
            current_positions
            - state[
                "last_position_physical"
            ][None, :]
            - global_displacement[None, :]
        )

        residual_error = np.linalg.norm(
            candidate_relative_displacements
            - expected_relative_displacement[
                None,
                :
            ],
            axis=1,
        )

        normalized_error = np.clip(
            residual_error
            / max(
                RELATIVE_MOTION_COST_SCALE_UM
                * max(frame_gap, 1),
                EPS,
            ),
            0.0,
            1.0,
        )

        # Confidence controls how strongly this term can influence
        # assignment. Uncertain relative motion contributes little.
        result[state_index] = (
            confidence * normalized_error
        )

    return result


def boundary_face_cost(
    states: list[dict],
    detections: pd.DataFrame,
) -> np.ndarray:
    """Penalize jumps between incompatible volume faces."""

    current_faces = [
        parse_boundary_faces(value)
        for value in detections["boundary_faces"]
    ]

    current_boundary = detections[
        "touches_boundary"
    ].to_numpy(dtype=bool)

    result = np.zeros(
        (len(states), len(detections)),
        dtype=float,
    )

    for state_index, state in enumerate(states):
        previous_faces = state[
            "last_boundary_faces"
        ]

        previous_boundary = bool(
            state["last_detection"].get(
                "touches_boundary",
                False,
            )
        ) or state["boundary_pending"]

        if not previous_boundary:
            continue

        for detection_index, faces in enumerate(
            current_faces
        ):
            # Moving from a boundary into the interior is valid.
            if not current_boundary[detection_index]:
                continue

            if previous_faces and faces:
                if previous_faces.isdisjoint(faces):
                    result[
                        state_index,
                        detection_index,
                    ] = 1.0

    return result


def student_t_scalar_cost(
    residual: np.ndarray,
    scale: np.ndarray | float,
    degrees_of_freedom: float,
) -> np.ndarray:
    """Heavy-tailed scalar negative-log cost, excluding constants."""

    residual = np.asarray(residual, dtype=float)
    scale = np.maximum(
        np.asarray(scale, dtype=float),
        EPS,
    )

    standardized_squared = (
        residual / scale
    ) ** 2

    return (
        0.5
        * (degrees_of_freedom + 1.0)
        * np.log1p(
            standardized_squared
            / degrees_of_freedom
        )
    )


def student_t_vector_cost(
    mahalanobis_squared: np.ndarray,
    degrees_of_freedom: float,
    dimension: int,
) -> np.ndarray:
    """Multivariate Student-t cost from squared Mahalanobis distance."""

    mahalanobis_squared = np.asarray(
        mahalanobis_squared,
        dtype=float,
    )

    return (
        0.5
        * (degrees_of_freedom + dimension)
        * np.log1p(
            mahalanobis_squared
            / degrees_of_freedom
        )
    )


def negative_log_probability(
    probability: np.ndarray | float,
) -> np.ndarray:
    """Stable -log(probability)."""

    return -np.log(
        np.clip(
            probability,
            PROBABILITY_FLOOR,
            1.0,
        )
    )


def track_position_sigma_zyx(
    *,
    state: dict,
    prediction_component: dict,
    global_shift_confidence: float,
) -> np.ndarray:
    """Adaptive physical position uncertainty for one track."""

    frame_gap = max(
        int(prediction_component["frame_gap"]),
        1,
    )

    sigma = (
        BASE_POSITION_SIGMA_ZYX_UM
        * np.sqrt(frame_gap)
    ).astype(float)

    sigma += (
        POSITION_SIGMA_GLOBAL_UNCERTAINTY_UM
        * (1.0 - np.clip(
            global_shift_confidence,
            0.0,
            1.0,
        ))
    )

    sigma += (
        POSITION_SIGMA_PER_MISSING_FRAME_UM
        * max(frame_gap - 1, 0)
    )

    if state["position_residual_samples"] > 0:
        sigma += (
            POSITION_SIGMA_TRACK_RESIDUAL_WEIGHT
            * state["position_residual_ema_zyx"]
        )

    if (
        state["boundary_pending"]
        or bool(
            state["last_detection"].get(
                "touches_boundary",
                False,
            )
        )
    ):
        sigma *= POSITION_SIGMA_BOUNDARY_SCALE

    return np.maximum(sigma, 0.25)


def track_miss_probability(
    *,
    state: dict,
    global_shift_confidence: float,
) -> float:
    """Context-dependent prior probability that a track is missed."""

    is_boundary_context = (
        state["boundary_pending"]
        or bool(
            state["last_detection"].get(
                "touches_boundary",
                False,
            )
        )
    )

    if state["missed_frames"] > 0:
        probability = (
            MISS_PROBABILITY_PENDING_BOUNDARY
            if is_boundary_context
            else MISS_PROBABILITY_PENDING_INTERIOR
        )
    else:
        probability = (
            MISS_PROBABILITY_BOUNDARY
            if is_boundary_context
            else MISS_PROBABILITY_INTERIOR
        )

    probability += (
        MISS_GLOBAL_UNCERTAINTY_BONUS
        * (1.0 - np.clip(
            global_shift_confidence,
            0.0,
            1.0,
        ))
    )

    return float(
        np.clip(
            probability,
            PROBABILITY_FLOOR,
            0.95,
        )
    )


def detection_birth_probability(
    detection: pd.Series,
) -> float:
    """Context-dependent prior probability that a detection is new."""

    probability = (
        BIRTH_PROBABILITY_BOUNDARY
        if bool(detection["touches_boundary"])
        else BIRTH_PROBABILITY_INTERIOR
    )

    return float(
        np.clip(
            probability,
            PROBABILITY_FLOOR,
            0.95,
        )
    )


def normalized_choice_probabilities(
    *,
    pair_costs: np.ndarray,
    alternative_cost: float,
    valid_mask: np.ndarray,
) -> tuple[np.ndarray, float]:
    """Normalize candidate scores against one miss/birth alternative."""

    pair_costs = np.asarray(pair_costs, dtype=float)
    valid_mask = np.asarray(valid_mask, dtype=bool)

    probabilities = np.zeros_like(
        pair_costs,
        dtype=float,
    )

    finite_costs = pair_costs[valid_mask]

    all_costs = np.concatenate(
        [
            finite_costs,
            np.asarray([alternative_cost]),
        ]
    )

    minimum_cost = float(np.min(all_costs))
    scores = np.exp(-(all_costs - minimum_cost))
    denominator = float(np.sum(scores))

    if finite_costs.size:
        probabilities[valid_mask] = (
            scores[:-1] / denominator
        )

    alternative_probability = float(
        scores[-1] / denominator
    )

    return probabilities, alternative_probability


def augmented_assignment(
    *,
    pair_cost_matrix: np.ndarray,
    miss_costs: np.ndarray,
    birth_costs: np.ndarray,
) -> dict:
    """Solve one-to-one pair/miss/birth assignment jointly."""

    track_count, detection_count = (
        pair_cost_matrix.shape
    )

    augmented_cost = np.full(
        (
            track_count + detection_count,
            detection_count + track_count,
        ),
        INVALID_COST,
        dtype=float,
    )

    augmented_cost[
        :track_count,
        :detection_count,
    ] = pair_cost_matrix

    # Each track has one private miss column.
    if track_count:
        augmented_cost[
            np.arange(track_count),
            detection_count + np.arange(track_count),
        ] = miss_costs

    # Each detection has one private birth row.
    if detection_count:
        augmented_cost[
            track_count + np.arange(detection_count),
            np.arange(detection_count),
        ] = birth_costs

    # Dummy birth rows and miss columns absorb one another at zero cost.
    augmented_cost[
        track_count:,
        detection_count:,
    ] = 0.0

    augmented_rows, augmented_cols = (
        linear_sum_assignment(
            augmented_cost
        )
    )

    pair_rows = []
    pair_cols = []
    missed_state_indices = []
    birth_detection_indices = []

    for row, col in zip(
        augmented_rows,
        augmented_cols,
    ):
        if row < track_count and col < detection_count:
            if pair_cost_matrix[row, col] < INVALID_COST:
                pair_rows.append(int(row))
                pair_cols.append(int(col))
            else:
                raise RuntimeError(
                    "Augmented assignment selected an invalid pair."
                )
        elif row < track_count and col >= detection_count:
            missed_state_indices.append(int(row))
        elif row >= track_count and col < detection_count:
            birth_detection_indices.append(int(col))

    return {
        "rows": np.asarray(pair_rows, dtype=int),
        "cols": np.asarray(pair_cols, dtype=int),
        "missed_state_indices": np.asarray(
            sorted(missed_state_indices),
            dtype=int,
        ),
        "birth_detection_indices": np.asarray(
            sorted(birth_detection_indices),
            dtype=int,
        ),
        "augmented_cost_matrix": augmented_cost,
        "augmented_rows": augmented_rows,
        "augmented_cols": augmented_cols,
        "objective_cost": float(
            np.sum(
                augmented_cost[
                    augmented_rows,
                    augmented_cols,
                ]
            )
        ),
    }

def assign_track_states(
    *,
    states: list[dict],
    detections: pd.DataFrame,
    current_frame: int,
    global_shift_history: dict[int, np.ndarray],
    global_shift_confidence: float,
):
    """Assign tracks using soft pair likelihoods and explicit miss/birth options."""

    track_count = len(states)
    detection_count = len(detections)

    if track_count == 0 or detection_count == 0:
        # Preserve explicit miss/birth decisions even when one side is empty.
        rows = np.array([], dtype=int)
        cols = np.array([], dtype=int)
        missed = np.arange(track_count, dtype=int)
        births = np.arange(detection_count, dtype=int)

        miss_probabilities = np.asarray(
            [
                track_miss_probability(
                    state=state,
                    global_shift_confidence=global_shift_confidence,
                )
                for state in states
            ],
            dtype=float,
        )

        birth_probabilities = np.asarray(
            [
                detection_birth_probability(
                    detections.iloc[index]
                )
                for index in range(detection_count)
            ],
            dtype=float,
        )

        return {
            "rows": rows,
            "cols": cols,
            "missed_state_indices": missed,
            "birth_detection_indices": births,
            "distance_matrix": np.empty((track_count, detection_count)),
            "position_sigma_zyx": np.empty((track_count, 3)),
            "mahalanobis_squared": np.empty((track_count, detection_count)),
            "volume_ratio": np.empty((track_count, detection_count)),
            "log_volume_change": np.empty((track_count, detection_count)),
            "pair_cost_matrix": np.empty((track_count, detection_count)),
            "cost_matrix": np.empty((track_count, detection_count)),
            "position_cost": np.empty((track_count, detection_count)),
            "volume_cost": np.empty((track_count, detection_count)),
            "size_cost": np.empty((track_count, detection_count)),
            "shape_cost": np.empty((track_count, detection_count)),
            "intensity_cost": np.empty((track_count, detection_count)),
            "bbox_cost": np.empty((track_count, detection_count)),
            "motion_cost": np.empty((track_count, detection_count)),
            "face_cost": np.empty((track_count, detection_count)),
            "safety_invalid": np.empty((track_count, detection_count), dtype=bool),
            "absolute_distance_limit": np.empty((track_count, detection_count)),
            "absolute_volume_ratio_limit": np.empty((track_count, detection_count)),
            "predicted_positions": np.empty((track_count, 3)),
            "global_only_positions": np.empty((track_count, 3)),
            "relative_motion_weights": np.empty(track_count),
            "boundary_related": np.empty((track_count, detection_count), dtype=bool),
            "prediction_components": [],
            "miss_probabilities": miss_probabilities,
            "miss_costs": negative_log_probability(miss_probabilities),
            "birth_probabilities": birth_probabilities,
            "birth_costs": negative_log_probability(birth_probabilities),
            "track_candidate_probabilities": np.empty((track_count, detection_count)),
            "detection_candidate_probabilities": np.empty((track_count, detection_count)),
            "association_probabilities": np.empty((track_count, detection_count)),
            "track_no_match_probabilities": miss_probabilities.copy(),
            "detection_birth_choice_probabilities": birth_probabilities.copy(),
            "track_probability_margins": np.zeros(track_count),
            "objective_cost": float(
                np.sum(negative_log_probability(miss_probabilities))
                + np.sum(negative_log_probability(birth_probabilities))
            ),
        }

    prediction_components = [
        predict_state_components(
            state=state,
            target_frame=current_frame,
            global_shift_history=global_shift_history,
        )
        for state in states
    ]

    predicted_positions = np.vstack(
        [
            item["predicted_position"]
            for item in prediction_components
        ]
    )

    global_only_positions = np.vstack(
        [
            state["last_position_physical"]
            + item["global_displacement"]
            for state, item in zip(
                states,
                prediction_components,
            )
        ]
    )

    relative_motion_weights = np.asarray(
        [
            item["relative_weight"]
            for item in prediction_components
        ],
        dtype=float,
    )

    current_positions = physical_coordinates(detections)

    residual_vectors = (
        current_positions[None, :, :]
        - predicted_positions[:, None, :]
    )

    distance_matrix = np.linalg.norm(
        residual_vectors,
        axis=2,
    )

    position_sigma_zyx = np.vstack(
        [
            track_position_sigma_zyx(
                state=state,
                prediction_component=component,
                global_shift_confidence=global_shift_confidence,
            )
            for state, component in zip(
                states,
                prediction_components,
            )
        ]
    )

    mahalanobis_squared = np.sum(
        (
            residual_vectors
            / position_sigma_zyx[:, None, :]
        ) ** 2,
        axis=2,
    )

    position_cost = student_t_vector_cost(
        mahalanobis_squared,
        POSITION_STUDENT_T_DOF,
        dimension=3,
    )

    frame_gaps = np.asarray(
        [
            current_frame - state["last_frame"]
            for state in states
        ],
        dtype=int,
    )

    previous_boundary = np.asarray(
        [
            bool(
                state["last_detection"].get(
                    "touches_boundary",
                    False,
                )
            )
            or state["boundary_pending"]
            for state in states
        ],
        dtype=bool,
    )

    current_boundary = detections[
        "touches_boundary"
    ].to_numpy(dtype=bool)

    boundary_related = (
        previous_boundary[:, None]
        | current_boundary[None, :]
    )

    previous_volume = np.asarray(
        [
            state_reference_value(
                state,
                "volume_voxels",
            )
            for state in states
        ],
        dtype=float,
    )

    current_volume = detections[
        "volume_voxels"
    ].to_numpy(dtype=float)

    safe_previous_volume = np.maximum(
        previous_volume,
        EPS,
    )
    safe_current_volume = np.maximum(
        current_volume,
        EPS,
    )

    log_volume_change = np.abs(
        np.log(
            safe_current_volume[None, :]
            / safe_previous_volume[:, None]
        )
    )

    volume_ratio = np.exp(log_volume_change)

    volume_scale = np.where(
        boundary_related,
        VOLUME_LOG_SCALE_BOUNDARY,
        VOLUME_LOG_SCALE_INTERIOR,
    )

    volume_cost = student_t_scalar_cost(
        log_volume_change,
        volume_scale,
        VOLUME_STUDENT_T_DOF,
    )

    size_residual = feature_group_cost(
        states,
        detections,
        SIZE_FEATURES,
    )
    shape_residual = feature_group_cost(
        states,
        detections,
        SHAPE_FEATURES,
    )
    intensity_residual = feature_group_cost(
        states,
        detections,
        INTENSITY_FEATURES,
    )
    bbox_residual = feature_group_cost(
        states,
        detections,
        BBOX_FEATURES,
    )

    size_cost = student_t_scalar_cost(
        size_residual,
        SIZE_RELATIVE_SCALE,
        FEATURE_STUDENT_T_DOF,
    )
    shape_cost = student_t_scalar_cost(
        shape_residual,
        SHAPE_RELATIVE_SCALE,
        FEATURE_STUDENT_T_DOF,
    )
    intensity_cost = student_t_scalar_cost(
        intensity_residual,
        INTENSITY_RELATIVE_SCALE,
        FEATURE_STUDENT_T_DOF,
    )
    bbox_cost = student_t_scalar_cost(
        bbox_residual,
        BBOX_RELATIVE_SCALE,
        FEATURE_STUDENT_T_DOF,
    )

    motion_cost = relative_motion_consistency_cost(
        states=states,
        current_positions=current_positions,
        current_frame=current_frame,
        global_shift_history=global_shift_history,
        prediction_components=prediction_components,
    )

    face_cost = boundary_face_cost(
        states,
        detections,
    )

    interior_cost = (
        INTERIOR_W_POSITION * position_cost
        + INTERIOR_W_VOLUME * volume_cost
        + INTERIOR_W_SIZE * size_cost
        + INTERIOR_W_SHAPE * shape_cost
        + INTERIOR_W_INTENSITY * intensity_cost
        + INTERIOR_W_BBOX * bbox_cost
        + INTERIOR_W_MOTION * motion_cost
    )

    boundary_cost = (
        BOUNDARY_W_POSITION * position_cost
        + BOUNDARY_W_VOLUME * volume_cost
        + BOUNDARY_W_MOTION * motion_cost
        + BOUNDARY_W_INTENSITY * intensity_cost
        + BOUNDARY_W_FACE * face_cost
    )

    pair_cost_matrix = np.where(
        boundary_related,
        boundary_cost,
        interior_cost,
    )

    absolute_distance_limit = np.where(
        boundary_related,
        (
            ABSOLUTE_MAX_DISTANCE_BOUNDARY_UM
            + ABSOLUTE_DISTANCE_PER_MISSING_FRAME_UM
            * np.maximum(frame_gaps[:, None] - 1, 0)
        ),
        (
            ABSOLUTE_MAX_DISTANCE_INTERIOR_UM
            + ABSOLUTE_DISTANCE_PER_MISSING_FRAME_UM
            * np.maximum(frame_gaps[:, None] - 1, 0)
        ),
    )

    absolute_volume_ratio_limit = np.where(
        boundary_related,
        ABSOLUTE_MAX_VOLUME_RATIO_BOUNDARY,
        ABSOLUTE_MAX_VOLUME_RATIO_INTERIOR,
    )

    safety_invalid = (
        (distance_matrix > absolute_distance_limit)
        | (volume_ratio > absolute_volume_ratio_limit)
        | ~np.isfinite(pair_cost_matrix)
    )

    pair_cost_matrix = pair_cost_matrix.copy()
    pair_cost_matrix[safety_invalid] = INVALID_COST

    miss_probabilities = np.asarray(
        [
            track_miss_probability(
                state=state,
                global_shift_confidence=global_shift_confidence,
            )
            for state in states
        ],
        dtype=float,
    )
    miss_costs = negative_log_probability(
        miss_probabilities
    )

    birth_probabilities = np.asarray(
        [
            detection_birth_probability(
                detections.iloc[index]
            )
            for index in range(detection_count)
        ],
        dtype=float,
    )
    birth_costs = negative_log_probability(
        birth_probabilities
    )

    assignment_result = augmented_assignment(
        pair_cost_matrix=pair_cost_matrix,
        miss_costs=miss_costs,
        birth_costs=birth_costs,
    )

    valid_pair_mask = ~safety_invalid

    track_candidate_probabilities = np.zeros_like(
        pair_cost_matrix,
        dtype=float,
    )
    track_no_match_probabilities = np.zeros(
        track_count,
        dtype=float,
    )

    for state_index in range(track_count):
        (
            track_candidate_probabilities[state_index],
            track_no_match_probabilities[state_index],
        ) = normalized_choice_probabilities(
            pair_costs=pair_cost_matrix[state_index],
            alternative_cost=float(miss_costs[state_index]),
            valid_mask=valid_pair_mask[state_index],
        )

    detection_candidate_probabilities = np.zeros_like(
        pair_cost_matrix,
        dtype=float,
    )
    detection_birth_choice_probabilities = np.zeros(
        detection_count,
        dtype=float,
    )

    for detection_index in range(detection_count):
        column_probabilities, birth_choice_probability = (
            normalized_choice_probabilities(
                pair_costs=pair_cost_matrix[:, detection_index],
                alternative_cost=float(birth_costs[detection_index]),
                valid_mask=valid_pair_mask[:, detection_index],
            )
        )

        detection_candidate_probabilities[
            :, detection_index
        ] = column_probabilities
        detection_birth_choice_probabilities[
            detection_index
        ] = birth_choice_probability

    association_probabilities = np.sqrt(
        track_candidate_probabilities
        * detection_candidate_probabilities
    )

    # Margin of the globally selected decision relative to that track's
    # best local alternative. It may be negative when the global one-to-one
    # solution deliberately chooses a locally second-best option to avoid a
    # worse conflict elsewhere.
    track_probability_margins = np.zeros(
        track_count,
        dtype=float,
    )

    selected_detection_by_state = {
        int(state_index): int(detection_index)
        for state_index, detection_index in zip(
            assignment_result["rows"],
            assignment_result["cols"],
        )
    }

    missed_state_set = set(
        assignment_result["missed_state_indices"].tolist()
    )

    for state_index in range(track_count):
        selected_detection = selected_detection_by_state.get(
            state_index
        )

        if selected_detection is not None:
            selected_probability = float(
                track_candidate_probabilities[
                    state_index,
                    selected_detection,
                ]
            )

            alternative_probabilities = np.concatenate(
                [
                    np.delete(
                        track_candidate_probabilities[state_index],
                        selected_detection,
                    ),
                    np.asarray([
                        track_no_match_probabilities[state_index]
                    ]),
                ]
            )
        elif state_index in missed_state_set:
            selected_probability = float(
                track_no_match_probabilities[state_index]
            )
            alternative_probabilities = (
                track_candidate_probabilities[state_index]
            )
        else:
            # Defensive fallback; every track row should be either matched
            # or assigned to its private miss column.
            selected_probability = 0.0
            alternative_probabilities = (
                track_candidate_probabilities[state_index]
            )

        best_alternative = (
            float(np.max(alternative_probabilities))
            if alternative_probabilities.size
            else 0.0
        )

        track_probability_margins[state_index] = (
            selected_probability - best_alternative
        )

    return {
        **assignment_result,
        "distance_matrix": distance_matrix,
        "position_sigma_zyx": position_sigma_zyx,
        "mahalanobis_squared": mahalanobis_squared,
        "volume_ratio": volume_ratio,
        "log_volume_change": log_volume_change,
        "pair_cost_matrix": pair_cost_matrix,
        # Compatibility alias for existing downstream code.
        "cost_matrix": pair_cost_matrix,
        "position_cost": position_cost,
        "volume_cost": volume_cost,
        "size_cost": size_cost,
        "shape_cost": shape_cost,
        "intensity_cost": intensity_cost,
        "bbox_cost": bbox_cost,
        "motion_cost": motion_cost,
        "face_cost": face_cost,
        "safety_invalid": safety_invalid,
        "absolute_distance_limit": absolute_distance_limit,
        "absolute_volume_ratio_limit": absolute_volume_ratio_limit,
        "predicted_positions": predicted_positions,
        "global_only_positions": global_only_positions,
        "relative_motion_weights": relative_motion_weights,
        "boundary_related": boundary_related,
        "prediction_components": prediction_components,
        "miss_probabilities": miss_probabilities,
        "miss_costs": miss_costs,
        "birth_probabilities": birth_probabilities,
        "birth_costs": birth_costs,
        "track_candidate_probabilities": track_candidate_probabilities,
        "detection_candidate_probabilities": detection_candidate_probabilities,
        "association_probabilities": association_probabilities,
        "track_no_match_probabilities": track_no_match_probabilities,
        "detection_birth_choice_probabilities": detection_birth_choice_probabilities,
        "track_probability_margins": track_probability_margins,
    }

def refine_global_shift_from_assignment(
    *,
    states: list[dict],
    detections: pd.DataFrame,
    current_frame: int,
    assignment: dict,
) -> dict:
    """Refine global shift using confident selected immediate-frame matches."""

    current_positions = physical_coordinates(
        detections
    )

    candidate_displacements = []
    candidate_pairs = []

    def collect_candidates(
        *,
        allow_boundary: bool,
    ) -> None:
        candidate_displacements.clear()
        candidate_pairs.clear()

        for state_index, detection_index in zip(
            assignment["rows"],
            assignment["cols"],
        ):
            state = states[int(state_index)]

            if state["last_frame"] != current_frame - 1:
                continue

            detection = detections.iloc[int(detection_index)]

            previous_boundary = bool(
                state["last_detection"].get(
                    "touches_boundary",
                    False,
                )
            )
            current_boundary = bool(
                detection["touches_boundary"]
            )

            if (
                not allow_boundary
                and (previous_boundary or current_boundary)
            ):
                continue

            association_probability = assignment[
                "association_probabilities"
            ][state_index, detection_index]

            probability_margin = assignment[
                "track_probability_margins"
            ][state_index]

            prediction_error = assignment[
                "distance_matrix"
            ][state_index, detection_index]

            if (
                association_probability
                < GLOBAL_REFINEMENT_MIN_ASSOCIATION_PROBABILITY
            ):
                continue

            if (
                probability_margin
                < GLOBAL_REFINEMENT_MIN_PROBABILITY_MARGIN
            ):
                continue

            if (
                prediction_error
                > GLOBAL_REFINEMENT_MAX_PREDICTION_ERROR_UM
            ):
                continue

            candidate_displacements.append(
                current_positions[int(detection_index)]
                - state["last_position_physical"]
            )
            candidate_pairs.append(
                (int(state_index), int(detection_index))
            )

    collect_candidates(allow_boundary=False)

    if len(candidate_displacements) < GLOBAL_REFINEMENT_MIN_MATCHES:
        collect_candidates(allow_boundary=True)

    if len(candidate_displacements) < GLOBAL_REFINEMENT_MIN_MATCHES:
        return {
            "accepted": False,
            "shift": np.zeros(3, dtype=float),
            "candidate_count": int(len(candidate_displacements)),
            "inlier_count": 0,
            "dispersion_um": np.nan,
            "confidence": 0.0,
        }

    summary = robust_displacement_summary(
        np.asarray(candidate_displacements, dtype=float)
    )

    accepted = (
        summary["inlier_count"]
        >= GLOBAL_REFINEMENT_MIN_MATCHES
    )

    return {
        "accepted": bool(accepted),
        "shift": summary["shift"],
        "candidate_count": int(len(candidate_displacements)),
        "inlier_count": summary["inlier_count"],
        "dispersion_um": summary["dispersion_um"],
        "confidence": summary["confidence"],
    }

def assignment_summary(
    assignment: dict,
) -> dict:
    """Return compact quality statistics for one probabilistic assignment."""

    rows = assignment["rows"]
    cols = assignment["cols"]

    if len(rows) == 0:
        median_distance = np.inf
        mean_pair_cost = np.inf
        median_probability = 0.0
    else:
        median_distance = float(
            np.median(
                assignment["distance_matrix"][rows, cols]
            )
        )
        mean_pair_cost = float(
            np.mean(
                assignment["pair_cost_matrix"][rows, cols]
            )
        )
        median_probability = float(
            np.median(
                assignment["association_probabilities"][rows, cols]
            )
        )

    decision_count = (
        len(assignment["rows"])
        + len(assignment["missed_state_indices"])
        + len(assignment["birth_detection_indices"])
    )

    normalized_objective = (
        assignment["objective_cost"]
        / max(decision_count, 1)
    )

    return {
        "matches": int(len(rows)),
        "misses": int(len(assignment["missed_state_indices"])),
        "births": int(len(assignment["birth_detection_indices"])),
        "median_distance_um": median_distance,
        "mean_pair_cost": mean_pair_cost,
        "median_association_probability": median_probability,
        "objective_cost": float(assignment["objective_cost"]),
        "normalized_objective": float(normalized_objective),
    }

def should_use_refined_assignment(
    *,
    initial_assignment: dict,
    refined_assignment: dict,
) -> bool:
    """Accept refinement when its joint pair/miss/birth objective improves."""

    initial = assignment_summary(initial_assignment)
    refined = assignment_summary(refined_assignment)

    if (
        refined["matches"]
        < initial["matches"] - GLOBAL_REFINEMENT_MAX_MATCH_LOSS
    ):
        return False

    if (
        refined["normalized_objective"]
        < initial["normalized_objective"]
        - GLOBAL_REFINEMENT_OBJECTIVE_TOLERANCE
    ):
        return True

    if refined["matches"] > initial["matches"]:
        return (
            refined["median_distance_um"]
            <= initial["median_distance_um"] + 0.50
        )

    return (
        refined["matches"] == initial["matches"]
        and refined["median_association_probability"]
        > initial["median_association_probability"]
        and refined["median_distance_um"]
        <= initial["median_distance_um"] + 0.25
    )

def update_matched_state(
    *,
    state: dict,
    detection: pd.Series,
    current_frame: int,
    global_shift_history: dict[int, np.ndarray],
    global_shift_confidence: float,
    predicted_position: np.ndarray,
    match_distance_um: float,
    match_cost: float,
    association_probability: float,
    probability_margin: float,
    boundary_related: bool,
) -> dict:
    """Update a selected match and conditionally learn motion/uncertainty."""

    previous_boundary = bool(
        state["last_detection"].get(
            "touches_boundary",
            False,
        )
    )

    was_reacquired = state["missed_frames"] > 0
    was_boundary_pending = bool(state["boundary_pending"])
    was_interior_pending = bool(state["interior_pending"])

    new_position_voxel = detection[
        ["centroid_z", "centroid_y", "centroid_x"]
    ].to_numpy(dtype=float)

    new_position_physical = (
        new_position_voxel * VOXEL_SIZE_ZYX
    )

    frame_gap = current_frame - state["last_frame"]

    global_displacement = cumulative_global_displacement(
        start_frame=state["last_frame"],
        target_frame=current_frame,
        global_shift_history=global_shift_history,
    )

    observed_relative_velocity = (
        new_position_physical
        - state["last_position_physical"]
        - global_displacement
    ) / max(frame_gap, 1)

    observed_relative_speed = float(
        np.linalg.norm(observed_relative_velocity)
    )

    current_boundary = bool(
        detection["touches_boundary"]
    )

    reliable_relative_update = (
        frame_gap == 1
        and not previous_boundary
        and not current_boundary
        and not boundary_related
        and global_shift_confidence
        >= RELATIVE_UPDATE_MIN_GLOBAL_CONFIDENCE
        and association_probability
        >= RELATIVE_UPDATE_MIN_ASSOCIATION_PROBABILITY
        and probability_margin
        >= RELATIVE_UPDATE_MIN_PROBABILITY_MARGIN
        and match_distance_um
        <= RELATIVE_UPDATE_MAX_DISTANCE_UM
        and match_cost
        <= RELATIVE_UPDATE_MAX_PAIR_COST
        and observed_relative_speed
        <= MAX_RELATIVE_VELOCITY_UM_PER_FRAME
    )

    relative_velocity_updated = False

    if reliable_relative_update:
        if state["relative_velocity_valid"]:
            innovation = float(
                np.linalg.norm(
                    observed_relative_velocity
                    - state["relative_velocity_physical"]
                )
            )

            state["relative_velocity_error_ema"] = (
                (1.0 - RELATIVE_ERROR_EMA_ALPHA)
                * state["relative_velocity_error_ema"]
                + RELATIVE_ERROR_EMA_ALPHA
                * innovation
            )

            state["relative_velocity_physical"] = (
                (1.0 - RELATIVE_VELOCITY_EMA_ALPHA)
                * state["relative_velocity_physical"]
                + RELATIVE_VELOCITY_EMA_ALPHA
                * observed_relative_velocity
            )
        else:
            state["relative_velocity_physical"] = (
                observed_relative_velocity
            )
            state["relative_velocity_valid"] = True
            state["relative_velocity_error_ema"] = 0.0

        state["relative_velocity_samples"] += 1
        state["last_relative_update_frame"] = int(current_frame)
        relative_velocity_updated = True
    else:
        state["relative_velocity_updates_rejected"] += 1

    prediction_residual_abs = np.abs(
        new_position_physical
        - np.asarray(predicted_position, dtype=float)
    )

    position_residual_updated = False

    if (
        frame_gap == 1
        and association_probability
        >= POSITION_RESIDUAL_UPDATE_MIN_ASSOCIATION_PROBABILITY
        and probability_margin
        >= POSITION_RESIDUAL_UPDATE_MIN_MARGIN
    ):
        if state["position_residual_samples"] == 0:
            state["position_residual_ema_zyx"] = (
                prediction_residual_abs
            )
        else:
            state["position_residual_ema_zyx"] = (
                (1.0 - POSITION_RESIDUAL_EMA_ALPHA)
                * state["position_residual_ema_zyx"]
                + POSITION_RESIDUAL_EMA_ALPHA
                * prediction_residual_abs
            )

        state["position_residual_samples"] += 1
        position_residual_updated = True

    state["previous_position_physical"] = (
        state["last_position_physical"].copy()
    )
    state["last_position_voxel"] = new_position_voxel
    state["last_position_physical"] = new_position_physical
    state["last_frame"] = int(current_frame)
    state["last_detection"] = detection.to_dict()
    state["missed_frames"] = 0
    state["boundary_pending"] = current_boundary
    state["interior_pending"] = False
    state["last_boundary_faces"] = parse_boundary_faces(
        detection["boundary_faces"]
    )

    update_feature_template(state, detection)

    return {
        "was_reacquired": was_reacquired,
        "was_boundary_pending": was_boundary_pending,
        "was_interior_pending": was_interior_pending,
        "previous_boundary": previous_boundary,
        "relative_velocity_updated": relative_velocity_updated,
        "position_residual_updated": position_residual_updated,
        "prediction_residual_abs_zyx": prediction_residual_abs,
        "observed_relative_velocity": observed_relative_velocity,
        "observed_relative_speed": observed_relative_speed,
        "global_displacement": global_displacement,
    }


In [ ]:
# ============================================================
# Annotate detections with boundary metadata
# ============================================================

time_frames = [
    annotate_boundary_metadata(frame)
    for frame in time_frames
]

boundary_counts = pd.DataFrame(
    {
        "frame": np.arange(len(time_frames)),
        "detections": [
            len(frame)
            for frame in time_frames
        ],
        "boundary_detections": [
            int(frame["touches_boundary"].sum())
            for frame in time_frames
        ],
    }
)

boundary_counts.head()


In [ ]:
# ============================================================
# Boundary-aware probabilistic tracking with global-relative motion
# ============================================================

track_states: dict[int, dict] = {}
track_records: list[dict] = []
boundary_events: list[dict] = []
missing_predictions: list[dict] = []
global_motion_records: list[dict] = []
tracking_diagnostic_records: list[dict] = []
association_event_records: list[dict] = []
association_candidate_records: list[dict] = []

# Maps target frame t to the global shift from t-1 -> t.
global_shift_history: dict[int, np.ndarray] = {}

next_track_id = 0


def append_track_record(
    *,
    track_id: int,
    frame: int,
    cell_index: int,
    detection: pd.Series,
    state: dict,
    match_type: str,
    match_distance_um: float | None,
    match_cost: float | None,
    association_probability: float | None,
    track_candidate_probability: float | None,
    detection_candidate_probability: float | None,
    probability_margin: float | None,
    position_cost: float | None,
    volume_cost: float | None,
    shape_cost: float | None,
    global_shift_physical: np.ndarray | None,
    global_shift_confidence: float | None,
    relative_velocity_updated: bool,
) -> None:
    """Append one observed detection to the output track table."""

    if global_shift_physical is None:
        global_shift_physical = np.full(3, np.nan, dtype=float)

    next_frame_relative_confidence = relative_motion_confidence(
        state,
        frame_gap=1,
    )

    track_records.append(
        {
            "track_id": int(track_id),
            "frame": int(frame),
            "cell": int(cell_index),
            "cell_id": int(detection.get("cell_id", cell_index)),
            "z": float(detection["centroid_z"]),
            "y": float(detection["centroid_y"]),
            "x": float(detection["centroid_x"]),
            "volume": float(detection["volume_voxels"]),
            "touches_boundary": bool(detection["touches_boundary"]),
            "boundary_faces": str(detection["boundary_faces"]),
            "distance_to_boundary_um": float(
                detection["distance_to_boundary_um"]
            ),
            "boundary_state": (
                "ACTIVE_BOUNDARY"
                if bool(detection["touches_boundary"])
                else "ACTIVE_INTERIOR"
            ),
            "match_type": str(match_type),
            "match_distance_um": (
                np.nan if match_distance_um is None
                else float(match_distance_um)
            ),
            "match_cost": (
                np.nan if match_cost is None
                else float(match_cost)
            ),
            "association_probability": (
                np.nan if association_probability is None
                else float(association_probability)
            ),
            "track_candidate_probability": (
                np.nan if track_candidate_probability is None
                else float(track_candidate_probability)
            ),
            "detection_candidate_probability": (
                np.nan if detection_candidate_probability is None
                else float(detection_candidate_probability)
            ),
            "probability_margin": (
                np.nan if probability_margin is None
                else float(probability_margin)
            ),
            "position_cost": (
                np.nan if position_cost is None
                else float(position_cost)
            ),
            "volume_cost": (
                np.nan if volume_cost is None
                else float(volume_cost)
            ),
            "shape_cost": (
                np.nan if shape_cost is None
                else float(shape_cost)
            ),
            "template_reliable": bool(state["template_reliable"]),
            "global_shift_z_um": float(global_shift_physical[0]),
            "global_shift_y_um": float(global_shift_physical[1]),
            "global_shift_x_um": float(global_shift_physical[2]),
            "global_shift_confidence": (
                np.nan if global_shift_confidence is None
                else float(global_shift_confidence)
            ),
            "relative_velocity_z_um_per_frame": float(
                state["relative_velocity_physical"][0]
            ),
            "relative_velocity_y_um_per_frame": float(
                state["relative_velocity_physical"][1]
            ),
            "relative_velocity_x_um_per_frame": float(
                state["relative_velocity_physical"][2]
            ),
            "relative_velocity_valid": bool(
                state["relative_velocity_valid"]
            ),
            "relative_velocity_samples": int(
                state["relative_velocity_samples"]
            ),
            "relative_velocity_error_ema_um": float(
                state["relative_velocity_error_ema"]
            ),
            "relative_motion_confidence_next_frame": float(
                next_frame_relative_confidence
            ),
            "relative_velocity_updated": bool(
                relative_velocity_updated
            ),
            "position_residual_ema_z_um": float(
                state["position_residual_ema_zyx"][0]
            ),
            "position_residual_ema_y_um": float(
                state["position_residual_ema_zyx"][1]
            ),
            "position_residual_ema_x_um": float(
                state["position_residual_ema_zyx"][2]
            ),
            "position_residual_samples": int(
                state["position_residual_samples"]
            ),
        }
    )


def safe_median_nearest_distance(
    distance_matrix: np.ndarray,
) -> float:
    """Median row-wise nearest distance, or NaN for an empty matrix."""

    if (
        distance_matrix.size == 0
        or distance_matrix.shape[0] == 0
        or distance_matrix.shape[1] == 0
    ):
        return np.nan

    return float(
        np.median(np.min(distance_matrix, axis=1))
    )


def record_top_candidates(
    *,
    assignment: dict,
    eligible_states: list[dict],
    detections: pd.DataFrame,
    current_frame: int,
) -> None:
    """Save the most relevant candidate pairs for later failure diagnosis."""

    if not SAVE_TOP_ASSOCIATION_CANDIDATES:
        return

    selected_by_state = {
        int(state_index): int(detection_index)
        for state_index, detection_index in zip(
            assignment["rows"],
            assignment["cols"],
        )
    }

    owner_by_detection = {
        int(detection_index): int(state_index)
        for state_index, detection_index in zip(
            assignment["rows"],
            assignment["cols"],
        )
    }

    for state_index, state in enumerate(eligible_states):
        valid_indices = np.flatnonzero(
            ~assignment["safety_invalid"][state_index]
        )

        ordered = valid_indices[
            np.argsort(
                assignment["pair_cost_matrix"][
                    state_index,
                    valid_indices,
                ]
            )
        ] if valid_indices.size else np.array([], dtype=int)

        keep = set(
            ordered[
                :TOP_ASSOCIATION_CANDIDATES_PER_TRACK
            ].tolist()
        )

        selected_detection = selected_by_state.get(state_index)
        if selected_detection is not None:
            keep.add(selected_detection)

        for rank, detection_index in enumerate(
            sorted(
                keep,
                key=lambda index: assignment[
                    "pair_cost_matrix"
                ][state_index, index],
            ),
            start=1,
        ):
            detection = detections.iloc[detection_index]
            owner_state_index = owner_by_detection.get(detection_index)

            association_candidate_records.append(
                {
                    "from_frame": int(state["last_frame"]),
                    "to_frame": int(current_frame),
                    "track_id": int(state["track_id"]),
                    "detection_position_index": int(detection_index),
                    "detection_cell_index": int(
                        detections.index[detection_index]
                    ),
                    "candidate_rank_by_pair_cost": int(rank),
                    "selected": bool(
                        selected_detection == detection_index
                    ),
                    "detection_selected_by_track_id": (
                        np.nan
                        if owner_state_index is None
                        else int(
                            eligible_states[
                                owner_state_index
                            ]["track_id"]
                        )
                    ),
                    "pair_cost": float(
                        assignment["pair_cost_matrix"][
                            state_index,
                            detection_index,
                        ]
                    ),
                    "association_probability": float(
                        assignment["association_probabilities"][
                            state_index,
                            detection_index,
                        ]
                    ),
                    "track_candidate_probability": float(
                        assignment["track_candidate_probabilities"][
                            state_index,
                            detection_index,
                        ]
                    ),
                    "detection_candidate_probability": float(
                        assignment["detection_candidate_probabilities"][
                            state_index,
                            detection_index,
                        ]
                    ),
                    "track_no_match_probability": float(
                        assignment["track_no_match_probabilities"][
                            state_index
                        ]
                    ),
                    "probability_margin": float(
                        assignment["track_probability_margins"][
                            state_index
                        ]
                    ),
                    "distance_um": float(
                        assignment["distance_matrix"][
                            state_index,
                            detection_index,
                        ]
                    ),
                    "mahalanobis_squared": float(
                        assignment["mahalanobis_squared"][
                            state_index,
                            detection_index,
                        ]
                    ),
                    "volume_ratio": float(
                        assignment["volume_ratio"][
                            state_index,
                            detection_index,
                        ]
                    ),
                    "log_volume_change": float(
                        assignment["log_volume_change"][
                            state_index,
                            detection_index,
                        ]
                    ),
                    "position_cost": float(
                        assignment["position_cost"][
                            state_index,
                            detection_index,
                        ]
                    ),
                    "volume_cost": float(
                        assignment["volume_cost"][
                            state_index,
                            detection_index,
                        ]
                    ),
                    "size_cost": float(
                        assignment["size_cost"][
                            state_index,
                            detection_index,
                        ]
                    ),
                    "shape_cost": float(
                        assignment["shape_cost"][
                            state_index,
                            detection_index,
                        ]
                    ),
                    "intensity_cost": float(
                        assignment["intensity_cost"][
                            state_index,
                            detection_index,
                        ]
                    ),
                    "bbox_cost": float(
                        assignment["bbox_cost"][
                            state_index,
                            detection_index,
                        ]
                    ),
                    "motion_cost": float(
                        assignment["motion_cost"][
                            state_index,
                            detection_index,
                        ]
                    ),
                    "face_cost": float(
                        assignment["face_cost"][
                            state_index,
                            detection_index,
                        ]
                    ),
                    "boundary_related": bool(
                        assignment["boundary_related"][
                            state_index,
                            detection_index,
                        ]
                    ),
                    "candidate_touches_boundary": bool(
                        detection["touches_boundary"]
                    ),
                }
            )


# ------------------------------------------------------------
# Initialize tracks from the first frame
# ------------------------------------------------------------

first_frame = time_frames[0]

for cell_index, detection in first_frame.iterrows():
    track_id = next_track_id
    next_track_id += 1

    state = make_track_state(
        track_id=track_id,
        frame=0,
        detection=detection,
    )
    track_states[track_id] = state

    match_type = (
        "boundary_entry"
        if bool(detection["touches_boundary"])
        else "initial"
    )

    append_track_record(
        track_id=track_id,
        frame=0,
        cell_index=int(cell_index),
        detection=detection,
        state=state,
        match_type=match_type,
        match_distance_um=None,
        match_cost=None,
        association_probability=None,
        track_candidate_probability=None,
        detection_candidate_probability=None,
        probability_margin=None,
        position_cost=None,
        volume_cost=None,
        shape_cost=None,
        global_shift_physical=None,
        global_shift_confidence=None,
        relative_velocity_updated=False,
    )

    if bool(detection["touches_boundary"]):
        boundary_events.append(
            {
                "track_id": track_id,
                "frame": 0,
                "event_type": "boundary_entry",
                "boundary_faces": detection["boundary_faces"],
                "missing_frames": 0,
                "reacquired_frame": np.nan,
                "confidence": np.nan,
            }
        )


# ------------------------------------------------------------
# Process each subsequent frame
# ------------------------------------------------------------

for current_frame in range(1, len(time_frames)):
    detections = time_frames[current_frame]

    eligible_states = []

    for state in track_states.values():
        if not state["active"]:
            continue

        frame_gap = current_frame - state["last_frame"]

        if frame_gap == 1:
            eligible_states.append(state)
            continue

        if (
            state["boundary_pending"]
            and frame_gap <= BOUNDARY_MAX_MISSING_FRAMES + 1
        ):
            eligible_states.append(state)
            continue

        if (
            state["interior_pending"]
            and frame_gap <= INTERIOR_MAX_MISSING_FRAMES + 1
        ):
            eligible_states.append(state)
            continue

        state["active"] = False

    previous_frame_positions = np.asarray(
        [
            state["last_position_physical"]
            for state in eligible_states
            if state["last_frame"] == current_frame - 1
        ],
        dtype=float,
    ).reshape(-1, 3)

    current_positions = physical_coordinates(detections)

    # --------------------------------------------------------
    # Pass 1: robust global shift and provisional assignment
    # --------------------------------------------------------

    initial_global_estimate = estimate_global_shift_physical(
        previous_frame_positions,
        current_positions,
    )

    initial_global_shift = initial_global_estimate["shift"]
    initial_global_confidence = float(
        initial_global_estimate["confidence"]
    )

    global_shift_history[current_frame] = (
        initial_global_shift.copy()
    )

    initial_assignment = assign_track_states(
        states=eligible_states,
        detections=detections,
        current_frame=current_frame,
        global_shift_history=global_shift_history,
        global_shift_confidence=initial_global_confidence,
    )

    # --------------------------------------------------------
    # Pass 2: refine global shift from confident assignments
    # --------------------------------------------------------

    refinement = refine_global_shift_from_assignment(
        states=eligible_states,
        detections=detections,
        current_frame=current_frame,
        assignment=initial_assignment,
    )

    refinement_used = False
    assignment = initial_assignment
    final_global_shift = initial_global_shift.copy()
    final_global_confidence = initial_global_confidence

    if refinement["accepted"]:
        global_shift_history[current_frame] = (
            refinement["shift"].copy()
        )

        refined_assignment = assign_track_states(
            states=eligible_states,
            detections=detections,
            current_frame=current_frame,
            global_shift_history=global_shift_history,
            global_shift_confidence=float(
                refinement["confidence"]
            ),
        )

        if should_use_refined_assignment(
            initial_assignment=initial_assignment,
            refined_assignment=refined_assignment,
        ):
            assignment = refined_assignment
            final_global_shift = refinement["shift"].copy()
            final_global_confidence = float(
                refinement["confidence"]
            )
            refinement_used = True
        else:
            global_shift_history[current_frame] = (
                initial_global_shift.copy()
            )

    previous_global_shift = global_shift_history.get(
        current_frame - 1,
        np.zeros(3, dtype=float),
    )

    direction_change_deg = (
        vector_angle_degrees(
            previous_global_shift,
            final_global_shift,
        )
        if current_frame > 1
        else np.nan
    )

    shift_change_um = float(
        np.linalg.norm(
            final_global_shift - previous_global_shift
        )
    )

    initial_assignment_stats = assignment_summary(
        initial_assignment
    )
    final_assignment_stats = assignment_summary(
        assignment
    )

    global_motion_records.append(
        {
            "from_frame": int(current_frame - 1),
            "to_frame": int(current_frame),
            "initial_method": initial_global_estimate["method"],
            "initial_pair_count": int(
                initial_global_estimate["pair_count"]
            ),
            "initial_inlier_count": int(
                initial_global_estimate["inlier_count"]
            ),
            "initial_dispersion_um": float(
                initial_global_estimate["dispersion_um"]
            ),
            "initial_confidence": initial_global_confidence,
            "initial_shift_z_um": float(initial_global_shift[0]),
            "initial_shift_y_um": float(initial_global_shift[1]),
            "initial_shift_x_um": float(initial_global_shift[2]),
            "refinement_accepted": bool(refinement["accepted"]),
            "refinement_used": bool(refinement_used),
            "refinement_candidate_count": int(
                refinement["candidate_count"]
            ),
            "refinement_inlier_count": int(
                refinement["inlier_count"]
            ),
            "refinement_dispersion_um": float(
                refinement["dispersion_um"]
            ),
            "refinement_confidence": float(
                refinement["confidence"]
            ),
            "final_shift_z_um": float(final_global_shift[0]),
            "final_shift_y_um": float(final_global_shift[1]),
            "final_shift_x_um": float(final_global_shift[2]),
            "final_shift_magnitude_um": float(
                np.linalg.norm(final_global_shift)
            ),
            "final_confidence": final_global_confidence,
            "shift_change_from_previous_um": shift_change_um,
            "direction_change_deg": float(direction_change_deg),
            "initial_matches": int(initial_assignment_stats["matches"]),
            "final_matches": int(final_assignment_stats["matches"]),
            "initial_misses": int(initial_assignment_stats["misses"]),
            "final_misses": int(final_assignment_stats["misses"]),
            "initial_births": int(initial_assignment_stats["births"]),
            "final_births": int(final_assignment_stats["births"]),
            "initial_normalized_objective": float(
                initial_assignment_stats["normalized_objective"]
            ),
            "final_normalized_objective": float(
                final_assignment_stats["normalized_objective"]
            ),
            "initial_median_match_distance_um": float(
                initial_assignment_stats["median_distance_um"]
            ),
            "final_median_match_distance_um": float(
                final_assignment_stats["median_distance_um"]
            ),
            "final_median_association_probability": float(
                final_assignment_stats[
                    "median_association_probability"
                ]
            ),
        }
    )

    record_top_candidates(
        assignment=assignment,
        eligible_states=eligible_states,
        detections=detections,
        current_frame=current_frame,
    )

    distance_matrix = assignment["distance_matrix"]
    global_only_distances = (
        cdist(
            assignment["global_only_positions"],
            current_positions,
        )
        if len(eligible_states) > 0 and len(detections) > 0
        else np.empty((len(eligible_states), len(detections)))
    )

    prediction_median_um = safe_median_nearest_distance(
        distance_matrix
    )
    global_only_median_um = safe_median_nearest_distance(
        global_only_distances
    )

    relative_weights = assignment["relative_motion_weights"]
    median_relative_weight = (
        float(np.median(relative_weights))
        if len(relative_weights) > 0
        else np.nan
    )

    rows = assignment["rows"]
    cols = assignment["cols"]
    missed_state_indices = assignment[
        "missed_state_indices"
    ]
    birth_detection_indices = assignment[
        "birth_detection_indices"
    ]

    matched_state_indices = set(rows.tolist())
    matched_detection_indices = set(cols.tolist())
    relative_updates_this_frame = 0

    # --------------------------------------------------------
    # Continue selected matches
    # --------------------------------------------------------

    for state_index, detection_index in zip(rows, cols):
        state = eligible_states[int(state_index)]
        detection = detections.iloc[int(detection_index)]

        previous_faces = "|".join(
            sorted(state["last_boundary_faces"])
        )

        distance_um = float(
            assignment["distance_matrix"][
                state_index,
                detection_index,
            ]
        )
        pair_cost = float(
            assignment["pair_cost_matrix"][
                state_index,
                detection_index,
            ]
        )
        association_probability = float(
            assignment["association_probabilities"][
                state_index,
                detection_index,
            ]
        )
        track_candidate_probability = float(
            assignment["track_candidate_probabilities"][
                state_index,
                detection_index,
            ]
        )
        detection_candidate_probability = float(
            assignment["detection_candidate_probabilities"][
                state_index,
                detection_index,
            ]
        )
        probability_margin = float(
            assignment["track_probability_margins"][
                state_index
            ]
        )
        is_boundary_related = bool(
            assignment["boundary_related"][
                state_index,
                detection_index,
            ]
        )

        update_result = update_matched_state(
            state=state,
            detection=detection,
            current_frame=current_frame,
            global_shift_history=global_shift_history,
            global_shift_confidence=final_global_confidence,
            predicted_position=assignment[
                "predicted_positions"
            ][state_index],
            match_distance_um=distance_um,
            match_cost=pair_cost,
            association_probability=association_probability,
            probability_margin=probability_margin,
            boundary_related=is_boundary_related,
        )

        relative_velocity_updated = bool(
            update_result["relative_velocity_updated"]
        )
        relative_updates_this_frame += int(
            relative_velocity_updated
        )

        if update_result["was_reacquired"]:
            if update_result["was_boundary_pending"]:
                match_type = "boundary_reacquired"
            else:
                match_type = "interior_reacquired"
        elif is_boundary_related:
            match_type = "boundary_partial"
        else:
            match_type = "normal"

        append_track_record(
            track_id=state["track_id"],
            frame=current_frame,
            cell_index=int(detections.index[detection_index]),
            detection=detection,
            state=state,
            match_type=match_type,
            match_distance_um=distance_um,
            match_cost=pair_cost,
            association_probability=association_probability,
            track_candidate_probability=track_candidate_probability,
            detection_candidate_probability=detection_candidate_probability,
            probability_margin=probability_margin,
            position_cost=float(
                assignment["position_cost"][
                    state_index,
                    detection_index,
                ]
            ),
            volume_cost=float(
                assignment["volume_cost"][
                    state_index,
                    detection_index,
                ]
            ),
            shape_cost=float(
                assignment["shape_cost"][
                    state_index,
                    detection_index,
                ]
            ),
            global_shift_physical=final_global_shift,
            global_shift_confidence=final_global_confidence,
            relative_velocity_updated=relative_velocity_updated,
        )

        association_event_records.append(
            {
                "from_frame": int(current_frame - 1),
                "to_frame": int(current_frame),
                "decision_type": "match",
                "track_id": int(state["track_id"]),
                "detection_position_index": int(detection_index),
                "detection_cell_index": int(
                    detections.index[detection_index]
                ),
                "pair_cost": pair_cost,
                "decision_cost": pair_cost,
                "association_probability": association_probability,
                "track_candidate_probability": track_candidate_probability,
                "detection_candidate_probability": detection_candidate_probability,
                "alternative_probability": float(
                    assignment["track_no_match_probabilities"][
                        state_index
                    ]
                ),
                "probability_margin": probability_margin,
                "distance_um": distance_um,
                "volume_ratio": float(
                    assignment["volume_ratio"][
                        state_index,
                        detection_index,
                    ]
                ),
                "position_cost": float(
                    assignment["position_cost"][
                        state_index,
                        detection_index,
                    ]
                ),
                "volume_cost": float(
                    assignment["volume_cost"][
                        state_index,
                        detection_index,
                    ]
                ),
                "shape_cost": float(
                    assignment["shape_cost"][
                        state_index,
                        detection_index,
                    ]
                ),
                "boundary_related": is_boundary_related,
            }
        )

        current_boundary = bool(detection["touches_boundary"])
        previous_boundary = bool(
            update_result["previous_boundary"]
        )

        if (
            update_result["was_reacquired"]
            and update_result["was_boundary_pending"]
        ):
            boundary_events.append(
                {
                    "track_id": state["track_id"],
                    "frame": current_frame,
                    "event_type": "boundary_reacquired",
                    "boundary_faces": detection["boundary_faces"],
                    "missing_frames": 0,
                    "reacquired_frame": current_frame,
                    "confidence": association_probability,
                }
            )
        elif previous_boundary and not current_boundary:
            boundary_events.append(
                {
                    "track_id": state["track_id"],
                    "frame": current_frame,
                    "event_type": "entered_interior",
                    "boundary_faces": previous_faces,
                    "missing_frames": 0,
                    "reacquired_frame": np.nan,
                    "confidence": association_probability,
                }
            )
        elif not previous_boundary and current_boundary:
            boundary_events.append(
                {
                    "track_id": state["track_id"],
                    "frame": current_frame,
                    "event_type": "boundary_exit_started",
                    "boundary_faces": detection["boundary_faces"],
                    "missing_frames": 0,
                    "reacquired_frame": np.nan,
                    "confidence": association_probability,
                }
            )

    # --------------------------------------------------------
    # Apply explicit miss decisions and preserve short memory
    # --------------------------------------------------------

    for state_index in missed_state_indices:
        state = eligible_states[int(state_index)]

        last_was_boundary = bool(
            state["last_detection"].get(
                "touches_boundary",
                False,
            )
        )
        boundary_context = (
            state["boundary_pending"]
            or last_was_boundary
        )

        state["missed_frames"] += 1

        if boundary_context:
            state["boundary_pending"] = True
            state["interior_pending"] = False
            pending_kind = "boundary"
            maximum_missing_frames = BOUNDARY_MAX_MISSING_FRAMES
        else:
            state["boundary_pending"] = False
            state["interior_pending"] = True
            pending_kind = "interior"
            maximum_missing_frames = INTERIOR_MAX_MISSING_FRAMES

        prediction = predict_state_components(
            state=state,
            target_frame=current_frame,
            global_shift_history=global_shift_history,
        )
        predicted_position = prediction["predicted_position"]

        missing_predictions.append(
            {
                "track_id": state["track_id"],
                "frame": current_frame,
                "pending_kind": pending_kind,
                "predicted_z": predicted_position[0] / VOXEL_SIZE_ZYX[0],
                "predicted_y": predicted_position[1] / VOXEL_SIZE_ZYX[1],
                "predicted_x": predicted_position[2] / VOXEL_SIZE_ZYX[2],
                "global_displacement_z_um": float(
                    prediction["global_displacement"][0]
                ),
                "global_displacement_y_um": float(
                    prediction["global_displacement"][1]
                ),
                "global_displacement_x_um": float(
                    prediction["global_displacement"][2]
                ),
                "relative_displacement_z_um": float(
                    prediction["relative_displacement"][0]
                ),
                "relative_displacement_y_um": float(
                    prediction["relative_displacement"][1]
                ),
                "relative_displacement_x_um": float(
                    prediction["relative_displacement"][2]
                ),
                "relative_motion_weight": float(
                    prediction["relative_weight"]
                ),
                "missed_frames": int(state["missed_frames"]),
                "miss_prior_probability": float(
                    assignment["miss_probabilities"][state_index]
                ),
                "miss_choice_probability": float(
                    assignment["track_no_match_probabilities"][
                        state_index
                    ]
                ),
                "miss_cost": float(
                    assignment["miss_costs"][state_index]
                ),
                "boundary_faces": "|".join(
                    sorted(state["last_boundary_faces"])
                ),
            }
        )

        association_event_records.append(
            {
                "from_frame": int(current_frame - 1),
                "to_frame": int(current_frame),
                "decision_type": "miss",
                "track_id": int(state["track_id"]),
                "detection_position_index": np.nan,
                "detection_cell_index": np.nan,
                "pair_cost": np.nan,
                "decision_cost": float(
                    assignment["miss_costs"][state_index]
                ),
                "association_probability": np.nan,
                "track_candidate_probability": np.nan,
                "detection_candidate_probability": np.nan,
                "alternative_probability": float(
                    assignment["track_no_match_probabilities"][
                        state_index
                    ]
                ),
                "probability_margin": float(
                    assignment["track_probability_margins"][
                        state_index
                    ]
                ),
                "distance_um": np.nan,
                "volume_ratio": np.nan,
                "position_cost": np.nan,
                "volume_cost": np.nan,
                "shape_cost": np.nan,
                "boundary_related": boundary_context,
            }
        )

        if boundary_context:
            boundary_events.append(
                {
                    "track_id": state["track_id"],
                    "frame": current_frame,
                    "event_type": "boundary_missing",
                    "boundary_faces": "|".join(
                        sorted(state["last_boundary_faces"])
                    ),
                    "missing_frames": state["missed_frames"],
                    "reacquired_frame": np.nan,
                    "confidence": float(
                        assignment["track_no_match_probabilities"][
                            state_index
                        ]
                    ),
                }
            )

        if state["missed_frames"] > maximum_missing_frames:
            state["active"] = False

            if boundary_context:
                boundary_events.append(
                    {
                        "track_id": state["track_id"],
                        "frame": current_frame,
                        "event_type": "boundary_exit_confirmed",
                        "boundary_faces": "|".join(
                            sorted(state["last_boundary_faces"])
                        ),
                        "missing_frames": state["missed_frames"],
                        "reacquired_frame": np.nan,
                        "confidence": np.nan,
                    }
                )

    # --------------------------------------------------------
    # Create tracks from explicit birth decisions
    # --------------------------------------------------------

    new_track_count = 0

    for detection_index in birth_detection_indices:
        detection = detections.iloc[int(detection_index)]

        track_id = next_track_id
        next_track_id += 1
        new_track_count += 1

        state = make_track_state(
            track_id=track_id,
            frame=current_frame,
            detection=detection,
        )
        track_states[track_id] = state

        is_boundary = bool(detection["touches_boundary"])
        match_type = (
            "boundary_entry" if is_boundary
            else "new_interior"
        )

        append_track_record(
            track_id=track_id,
            frame=current_frame,
            cell_index=int(detections.index[detection_index]),
            detection=detection,
            state=state,
            match_type=match_type,
            match_distance_um=None,
            match_cost=float(
                assignment["birth_costs"][detection_index]
            ),
            association_probability=None,
            track_candidate_probability=None,
            detection_candidate_probability=None,
            probability_margin=None,
            position_cost=None,
            volume_cost=None,
            shape_cost=None,
            global_shift_physical=final_global_shift,
            global_shift_confidence=final_global_confidence,
            relative_velocity_updated=False,
        )

        association_event_records.append(
            {
                "from_frame": int(current_frame - 1),
                "to_frame": int(current_frame),
                "decision_type": "birth",
                "track_id": int(track_id),
                "detection_position_index": int(detection_index),
                "detection_cell_index": int(
                    detections.index[detection_index]
                ),
                "pair_cost": np.nan,
                "decision_cost": float(
                    assignment["birth_costs"][detection_index]
                ),
                "association_probability": np.nan,
                "track_candidate_probability": np.nan,
                "detection_candidate_probability": np.nan,
                "alternative_probability": float(
                    assignment[
                        "detection_birth_choice_probabilities"
                    ][detection_index]
                ),
                "probability_margin": np.nan,
                "distance_um": np.nan,
                "volume_ratio": np.nan,
                "position_cost": np.nan,
                "volume_cost": np.nan,
                "shape_cost": np.nan,
                "boundary_related": is_boundary,
            }
        )

        if is_boundary:
            boundary_events.append(
                {
                    "track_id": track_id,
                    "frame": current_frame,
                    "event_type": "boundary_entry",
                    "boundary_faces": detection["boundary_faces"],
                    "missing_frames": 0,
                    "reacquired_frame": np.nan,
                    "confidence": float(
                        assignment[
                            "detection_birth_choice_probabilities"
                        ][detection_index]
                    ),
                }
            )

    valid_candidate_tracks = int(
        (~assignment["safety_invalid"]).any(axis=1).sum()
    ) if len(eligible_states) and len(detections) else 0

    matched_probabilities = (
        assignment["association_probabilities"][rows, cols]
        if len(rows)
        else np.asarray([], dtype=float)
    )

    tracking_diagnostic_records.append(
        {
            "from_frame": int(current_frame - 1),
            "to_frame": int(current_frame),
            "eligible_tracks": int(len(eligible_states)),
            "detections": int(len(detections)),
            "matches": int(len(rows)),
            "selected_misses": int(len(missed_state_indices)),
            "selected_births": int(len(birth_detection_indices)),
            "new_tracks": int(new_track_count),
            "tracks_with_safety_valid_candidate": valid_candidate_tracks,
            "median_nearest_prediction_distance_um": prediction_median_um,
            "median_nearest_global_only_distance_um": global_only_median_um,
            "median_relative_motion_weight": median_relative_weight,
            "median_selected_association_probability": (
                float(np.median(matched_probabilities))
                if matched_probabilities.size else np.nan
            ),
            "minimum_selected_association_probability": (
                float(np.min(matched_probabilities))
                if matched_probabilities.size else np.nan
            ),
            "normalized_assignment_objective": float(
                final_assignment_stats["normalized_objective"]
            ),
            "relative_velocity_updates": int(
                relative_updates_this_frame
            ),
            "global_shift_confidence": float(
                final_global_confidence
            ),
            "global_direction_change_deg": float(
                direction_change_deg
            ),
            "global_shift_change_um": float(shift_change_um),
            "refinement_used": bool(refinement_used),
        }
    )

    print(
        f"\nDiagnostics {current_frame - 1:03d}"
        f"->{current_frame:03d}"
    )
    print(
        "Final global shift ZYX (µm):",
        np.round(final_global_shift, 3),
        "| magnitude:",
        round(float(np.linalg.norm(final_global_shift)), 3),
        "| confidence:",
        round(final_global_confidence, 3),
    )

    if current_frame > 1:
        print(
            "Global direction change:",
            "nan" if np.isnan(direction_change_deg)
            else round(direction_change_deg, 1),
            "degrees | shift-vector change:",
            round(shift_change_um, 3),
            "µm",
        )

    print(
        "Global refinement:",
        "used" if refinement_used else "not used",
        "| candidates:",
        refinement["candidate_count"],
        "| initial/final matches:",
        initial_assignment_stats["matches"],
        "/",
        final_assignment_stats["matches"],
    )
    print(
        "Median nearest distance using final/global-only predictor:",
        "nan" if np.isnan(prediction_median_um)
        else round(prediction_median_um, 3),
        "/",
        "nan" if np.isnan(global_only_median_um)
        else round(global_only_median_um, 3),
        "µm",
    )
    print(
        "Decisions:",
        len(rows),
        "matches |",
        len(missed_state_indices),
        "misses |",
        len(birth_detection_indices),
        "births",
    )
    print(
        "Median selected association probability:",
        "nan" if not matched_probabilities.size
        else round(float(np.median(matched_probabilities)), 3),
        "| normalized objective:",
        round(final_assignment_stats["normalized_objective"], 3),
    )
    print(
        "Tracks with at least one broad-safety-valid candidate:",
        valid_candidate_tracks,
        "/",
        len(eligible_states),
        "| reliable velocity updates:",
        relative_updates_this_frame,
    )


tracks = pd.DataFrame(track_records)
boundary_events = pd.DataFrame(boundary_events)
missing_predictions = pd.DataFrame(missing_predictions)
boundary_predictions = (
    missing_predictions[
        missing_predictions["pending_kind"] == "boundary"
    ].copy()
    if not missing_predictions.empty
    else pd.DataFrame()
)
global_motion = pd.DataFrame(global_motion_records)
tracking_diagnostics = pd.DataFrame(
    tracking_diagnostic_records
)
association_events = pd.DataFrame(
    association_event_records
)
association_candidates = pd.DataFrame(
    association_candidate_records
)


In [ ]:
# ============================================================
# Tracking summary
# ============================================================

summary = {
    "track_records": len(tracks),
    "unique_tracks": tracks["track_id"].nunique(),
    "boundary_track_records": int(
        tracks["touches_boundary"].sum()
    ),
    "boundary_reacquisitions": int(
        (
            boundary_events["event_type"]
            == "boundary_reacquired"
        ).sum()
    ) if not boundary_events.empty else 0,
    "interior_reacquisitions": int(
        (tracks["match_type"] == "interior_reacquired").sum()
    ),
    "confirmed_boundary_exits": int(
        (
            boundary_events["event_type"]
            == "boundary_exit_confirmed"
        ).sum()
    ) if not boundary_events.empty else 0,
    "selected_misses": int(
        (association_events["decision_type"] == "miss").sum()
    ) if not association_events.empty else 0,
    "selected_births": int(
        (association_events["decision_type"] == "birth").sum()
    ) if not association_events.empty else 0,
    "median_association_probability": float(
        tracks["association_probability"].median()
    ),
    "relative_velocity_updates": int(
        tracks["relative_velocity_updated"].sum()
    ),
    "tracks_with_relative_velocity": int(
        sum(
            state["relative_velocity_valid"]
            for state in track_states.values()
        )
    ),
    "mean_global_shift_confidence": float(
        global_motion["final_confidence"].mean()
    ) if not global_motion.empty else np.nan,
    "largest_global_direction_change_deg": float(
        global_motion["direction_change_deg"].max()
    ) if not global_motion.empty else np.nan,
    "global_refinements_used": int(
        global_motion["refinement_used"].sum()
    ) if not global_motion.empty else 0,
}

pd.Series(summary)


## Save probabilistic global-relative tracking results

`tracks.csv` contains observed detections, selected association confidence, the
main evidence costs, and the global-relative motion state associated with every
accepted continuation.

Additional audit files:

- `association_events.csv`: every selected match, miss, and birth decision;
- `association_candidates.csv`: the top candidate pairs per track, including
  position, volume, morphology, probability, competition, and ownership data;
- `missing_predictions.csv`: predicted positions and miss probabilities for
  both interior- and boundary-pending tracks;
- `boundary_predictions.csv`: the boundary-only subset retained for backward
  compatibility;
- `global_motion.csv`: initial and refined frame shifts, confidence, direction
  changes, and joint assignment objectives;
- `tracking_diagnostics.csv`: per-transition match/miss/birth counts,
  probabilities, prediction errors, and relative-motion usage;
- `track_states.csv`: final persistent state and learned uncertainty of every
  track.


In [ ]:
# ============================================================
# Save probabilistic global-relative tracking results
# ============================================================

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

tracks.to_csv(OUTPUT_DIR / "tracks.csv", index=False)
boundary_events.to_csv(
    OUTPUT_DIR / "boundary_events.csv",
    index=False,
)
boundary_predictions.to_csv(
    OUTPUT_DIR / "boundary_predictions.csv",
    index=False,
)
missing_predictions.to_csv(
    OUTPUT_DIR / "missing_predictions.csv",
    index=False,
)
boundary_counts.to_csv(
    OUTPUT_DIR / "boundary_detection_counts.csv",
    index=False,
)
global_motion.to_csv(
    OUTPUT_DIR / "global_motion.csv",
    index=False,
)
tracking_diagnostics.to_csv(
    OUTPUT_DIR / "tracking_diagnostics.csv",
    index=False,
)
association_events.to_csv(
    OUTPUT_DIR / "association_events.csv",
    index=False,
)
association_candidates.to_csv(
    OUTPUT_DIR / "association_candidates.csv",
    index=False,
)

final_track_states = pd.DataFrame(
    [
        {
            "track_id": state["track_id"],
            "active": state["active"],
            "last_frame": state["last_frame"],
            "missed_frames": state["missed_frames"],
            "boundary_pending": state["boundary_pending"],
            "interior_pending": state["interior_pending"],
            "last_boundary_faces": "|".join(
                sorted(state["last_boundary_faces"])
            ),
            "template_reliable": state["template_reliable"],
            "template_count": state["template_count"],
            "relative_velocity_valid": state[
                "relative_velocity_valid"
            ],
            "relative_velocity_samples": state[
                "relative_velocity_samples"
            ],
            "relative_velocity_error_ema_um": state[
                "relative_velocity_error_ema"
            ],
            "relative_velocity_updates_rejected": state[
                "relative_velocity_updates_rejected"
            ],
            "last_relative_update_frame": state[
                "last_relative_update_frame"
            ],
            "relative_velocity_z_um_per_frame": float(
                state["relative_velocity_physical"][0]
            ),
            "relative_velocity_y_um_per_frame": float(
                state["relative_velocity_physical"][1]
            ),
            "relative_velocity_x_um_per_frame": float(
                state["relative_velocity_physical"][2]
            ),
            "relative_motion_confidence_next_frame": (
                relative_motion_confidence(state, frame_gap=1)
            ),
            "position_residual_ema_z_um": float(
                state["position_residual_ema_zyx"][0]
            ),
            "position_residual_ema_y_um": float(
                state["position_residual_ema_zyx"][1]
            ),
            "position_residual_ema_x_um": float(
                state["position_residual_ema_zyx"][2]
            ),
            "position_residual_samples": int(
                state["position_residual_samples"]
            ),
        }
        for state in track_states.values()
    ]
).sort_values("track_id")

final_track_states.to_csv(
    OUTPUT_DIR / "track_states.csv",
    index=False,
)

metadata = {
    "architecture": (
        "boundary_aware_probabilistic_global_relative_motion_tracking"
    ),
    "association_probabilities_are_calibrated": False,
    "sample_id": SAMPLE_ID,
    "volume_shape_zyx": VOLUME_SHAPE_ZYX.tolist(),
    "voxel_size_zyx": VOXEL_SIZE_ZYX.tolist(),
    "global_shift": {
        "max_pair_distance_um": GLOBAL_SHIFT_MAX_PAIR_DISTANCE_UM,
        "mad_scale": GLOBAL_SHIFT_MAD_SCALE,
        "min_inlier_radius_um": GLOBAL_SHIFT_MIN_INLIER_RADIUS_UM,
        "confidence_pair_count": GLOBAL_SHIFT_CONFIDENCE_PAIR_COUNT,
        "confidence_dispersion_um": GLOBAL_SHIFT_CONFIDENCE_DISPERSION_UM,
        "refinement_min_matches": GLOBAL_REFINEMENT_MIN_MATCHES,
        "refinement_min_association_probability": (
            GLOBAL_REFINEMENT_MIN_ASSOCIATION_PROBABILITY
        ),
        "refinement_min_probability_margin": (
            GLOBAL_REFINEMENT_MIN_PROBABILITY_MARGIN
        ),
        "refinement_max_prediction_error_um": (
            GLOBAL_REFINEMENT_MAX_PREDICTION_ERROR_UM
        ),
    },
    "position_likelihood": {
        "base_sigma_zyx_um": BASE_POSITION_SIGMA_ZYX_UM.tolist(),
        "global_uncertainty_um": (
            POSITION_SIGMA_GLOBAL_UNCERTAINTY_UM
        ),
        "per_missing_frame_um": (
            POSITION_SIGMA_PER_MISSING_FRAME_UM
        ),
        "boundary_scale": POSITION_SIGMA_BOUNDARY_SCALE,
        "track_residual_weight": (
            POSITION_SIGMA_TRACK_RESIDUAL_WEIGHT
        ),
        "student_t_degrees_of_freedom": POSITION_STUDENT_T_DOF,
    },
    "volume_likelihood": {
        "interior_log_scale": VOLUME_LOG_SCALE_INTERIOR,
        "boundary_log_scale": VOLUME_LOG_SCALE_BOUNDARY,
        "student_t_degrees_of_freedom": VOLUME_STUDENT_T_DOF,
    },
    "miss_priors": {
        "interior": MISS_PROBABILITY_INTERIOR,
        "boundary": MISS_PROBABILITY_BOUNDARY,
        "pending_interior": MISS_PROBABILITY_PENDING_INTERIOR,
        "pending_boundary": MISS_PROBABILITY_PENDING_BOUNDARY,
        "global_uncertainty_bonus": MISS_GLOBAL_UNCERTAINTY_BONUS,
    },
    "birth_priors": {
        "interior": BIRTH_PROBABILITY_INTERIOR,
        "boundary": BIRTH_PROBABILITY_BOUNDARY,
    },
    "broad_safety_gates": {
        "max_distance_interior_um": (
            ABSOLUTE_MAX_DISTANCE_INTERIOR_UM
        ),
        "max_distance_boundary_um": (
            ABSOLUTE_MAX_DISTANCE_BOUNDARY_UM
        ),
        "distance_per_missing_frame_um": (
            ABSOLUTE_DISTANCE_PER_MISSING_FRAME_UM
        ),
        "max_volume_ratio_interior": (
            ABSOLUTE_MAX_VOLUME_RATIO_INTERIOR
        ),
        "max_volume_ratio_boundary": (
            ABSOLUTE_MAX_VOLUME_RATIO_BOUNDARY
        ),
    },
    "memory": {
        "interior_max_missing_frames": INTERIOR_MAX_MISSING_FRAMES,
        "boundary_max_missing_frames": BOUNDARY_MAX_MISSING_FRAMES,
    },
    "relative_motion": {
        "velocity_ema_alpha": RELATIVE_VELOCITY_EMA_ALPHA,
        "error_ema_alpha": RELATIVE_ERROR_EMA_ALPHA,
        "full_confidence_samples": RELATIVE_FULL_CONFIDENCE_SAMPLES,
        "error_confidence_scale_um": RELATIVE_ERROR_CONFIDENCE_SCALE_UM,
        "gap_decay": RELATIVE_MOTION_GAP_DECAY,
        "boundary_confidence_scale": (
            BOUNDARY_RELATIVE_MOTION_CONFIDENCE_SCALE
        ),
        "update_min_global_confidence": (
            RELATIVE_UPDATE_MIN_GLOBAL_CONFIDENCE
        ),
        "update_min_association_probability": (
            RELATIVE_UPDATE_MIN_ASSOCIATION_PROBABILITY
        ),
        "update_min_probability_margin": (
            RELATIVE_UPDATE_MIN_PROBABILITY_MARGIN
        ),
        "update_max_distance_um": RELATIVE_UPDATE_MAX_DISTANCE_UM,
        "max_velocity_um_per_frame": (
            MAX_RELATIVE_VELOCITY_UM_PER_FRAME
        ),
    },
    "boundary_margin_um": BOUNDARY_MARGIN_UM,
    "candidate_diagnostics": {
        "saved": SAVE_TOP_ASSOCIATION_CANDIDATES,
        "top_per_track": TOP_ASSOCIATION_CANDIDATES_PER_TRACK,
    },
    "track_records": int(len(tracks)),
    "unique_tracks": int(tracks["track_id"].nunique()),
    "association_events": int(len(association_events)),
    "association_candidates": int(len(association_candidates)),
    "selected_misses": int(
        (association_events["decision_type"] == "miss").sum()
    ) if not association_events.empty else 0,
    "selected_births": int(
        (association_events["decision_type"] == "birth").sum()
    ) if not association_events.empty else 0,
    "boundary_events": int(len(boundary_events)),
    "boundary_reacquisitions": int(
        (
            boundary_events["event_type"]
            == "boundary_reacquired"
        ).sum()
    ) if not boundary_events.empty else 0,
    "interior_reacquisitions": int(
        (tracks["match_type"] == "interior_reacquired").sum()
    ),
    "relative_velocity_updates": int(
        tracks["relative_velocity_updated"].sum()
    ),
    "global_refinements_used": int(
        global_motion["refinement_used"].sum()
    ) if not global_motion.empty else 0,
}

with open(
    OUTPUT_DIR / "metadata.json",
    "w",
    encoding="utf-8",
) as file:
    json.dump(metadata, file, indent=4)

print(f"Saved {len(tracks):,} track records.")
print(f"Output directory: {OUTPUT_DIR}")
print()
print("Files:")
for output_file in sorted(OUTPUT_DIR.iterdir()):
    print(" ", output_file.name)
